In [3]:
import os
import re
import csv
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datetime import datetime
from ultralytics import YOLO
from collections import defaultdict
import pyttsx3
import json
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns
from matplotlib import cm

# clustering helpers
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Transformer imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

class DotGridHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, 1, 1)

    def forward(self, feat):
        hmap = self.conv(feat)  # Bx1xhxw
        grid_logits = F.adaptive_avg_pool2d(hmap, (3, 2))  # Bx1x3x2
        return hmap, grid_logits

class ClassFromGridHead(nn.Module):
    def __init__(self, num_classes=26):
        super().__init__()
        self.fc = nn.Linear(6, num_classes)

    def forward(self, grid_probs):
        x = grid_probs.view(grid_probs.size(0), -1)
        return self.fc(x)

class BrailleXAI(nn.Module):
    def __init__(self, num_classes=26):
        super().__init__()
        # Adjusted for 96x64 input images
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
        )
        self.dot_head = DotGridHead(128)
        self.class_from_grid = ClassFromGridHead(num_classes)

    def forward(self, x):
        feat = self.backbone(x)  # Bx128x24x16 (for 96x64 input)
        hmap, grid_logits = self.dot_head(feat)
        grid_probs = torch.sigmoid(grid_logits)
        class_logits = self.class_from_grid(grid_probs)
        return {
            "class_logits": class_logits,
            "grid_logits": grid_logits,
            "grid_probs": grid_probs,
            "heatmap": hmap
        }

def load_attention_model(model_path, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Load the trained BrailleXAI model"""
    model = BrailleXAI(num_classes=26)  # 26 letters
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

def process_crop_with_attention(model, crop_img, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Process a single crop with the BrailleXAI model"""
    # Convert to grayscale if needed
    if len(crop_img.shape) == 3:
        gray = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)
    else:
        gray = crop_img
    
    # Resize to expected input size (64x96 as used in training)
    resized = cv2.resize(gray, (64, 96))  # Width, Height
    
    # Normalize and convert to tensor
    tensor_img = torch.from_numpy(resized).float() / 255.0
    tensor_img = tensor_img.unsqueeze(0).unsqueeze(0)  # Add batch and channel dims
    tensor_img = tensor_img.to(device)
    
    # Run through model
    with torch.no_grad():
        outputs = model(tensor_img)
    
    # Process outputs
    class_probs = F.softmax(outputs["class_logits"], dim=1)
    class_conf, class_idx = torch.max(class_probs, dim=1)
    
    # Map class index to character (0-25: a-z)
    attn_label = chr(ord('a') + class_idx.item())
    
    # Get grid pattern
    grid_pattern = (outputs["grid_probs"] > 0.5).int().squeeze().cpu().numpy().flatten().tolist()
    
    # Get heatmap
    heatmap = outputs["heatmap"].squeeze().cpu().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = cv2.resize(heatmap, (crop_img.shape[1], crop_img.shape[0]))
    heatmap = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # Calculate dot count from grid pattern
    dot_count = sum(grid_pattern)
    
    # Get class probabilities for all classes
    all_probs = class_probs.squeeze().cpu().numpy()
    
    return {
        "label": attn_label,
        "confidence": class_conf.item(),
        "grid_pattern": grid_pattern,
        "heatmap": heatmap,
        "dot_count": dot_count,
        "all_probs": all_probs,
        "class_idx": class_idx.item()
    }

def generate_detailed_explanation(attn_results, yolo_label, yolo_conf):
    """Generate detailed explanation text based on attention results"""
    grid = attn_results["grid_pattern"]
    label = attn_results["label"]
    conf = attn_results["confidence"]
    dot_count = attn_results["dot_count"]
    
    # Describe the grid pattern
    dot_positions = []
    positions = ["top-left", "middle-left", "bottom-left", "top-right", "middle-right", "bottom-right"]
    
    for i, has_dot in enumerate(grid):
        if has_dot:
            dot_positions.append(positions[i])
    
    # Start building the explanation
    explanation_parts = []
    
    # Basic detection info
    explanation_parts.append(f"Character detected as '{label}' with {conf:.1%} confidence.")
    
    # Dot pattern explanation
    if dot_positions:
        dots_desc = ", ".join(dot_positions)
        explanation_parts.append(f"Detected dots at: {dots_desc}.")
        explanation_parts.append(f"Total dots: {dot_count}.")
    else:
        explanation_parts.append("No dots detected.")
    
    # Compare with YOLO prediction
    if yolo_label.lower() != label and yolo_label != "?":
        explanation_parts.append(f"Disagrees with YOLO prediction '{yolo_label}' (confidence: {yolo_conf:.1%}).")
        
        # Provide possible reasons for disagreement
        if dot_count == 0 and yolo_label != " ":
            explanation_parts.append("Possible reason: YOLO detected a character but no dots were found.")
        elif dot_count > 0 and yolo_label == " ":
            explanation_parts.append("Possible reason: Dots were detected but YOLO predicted a space.")
        else:
            explanation_parts.append("Possible reason: Different interpretation of dot pattern.")
    else:
        explanation_parts.append(f"Confirms YOLO prediction '{yolo_label}' (confidence: {yolo_conf:.1%}).")
    
    # Add confidence assessment
    if conf > 0.9:
        explanation_parts.append("High confidence in this prediction.")
    elif conf > 0.7:
        explanation_parts.append("Moderate confidence in this prediction.")
    else:
        explanation_parts.append("Low confidence in this prediction. Consider manual verification.")
    
    return " ".join(explanation_parts)

def create_comprehensive_visualization(crop_img, attn_results, output_path, yolo_label, yolo_conf):
    """Create comprehensive visualization for a single character"""
    os.makedirs(output_path, exist_ok=True)
    
    # Create a figure with multiple subplots
    fig = plt.figure(figsize=(20, 12))
    
    # 1. Original crop
    ax1 = plt.subplot(2, 3, 1)
    if len(crop_img.shape) == 3:
        plt.imshow(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(crop_img, cmap='gray')
    plt.title(f'Original Crop\nYOLO: {yolo_label} ({yolo_conf:.1%})')
    plt.axis('off')
    
    # 2. Heatmap overlay
    ax2 = plt.subplot(2, 3, 2)
    heatmap = attn_results["heatmap"]
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(crop_img, 0.6, heatmap_color, 0.4, 0)
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title('Attention Heatmap Overlay')
    plt.axis('off')
    
    # 3. Grid visualization
    ax3 = plt.subplot(2, 3, 3)
    grid_img = draw_grid_on_image(crop_img, attn_results["grid_pattern"])
    plt.imshow(cv2.cvtColor(grid_img, cv2.COLOR_BGR2RGB))
    plt.title('Detected Dot Pattern')
    plt.axis('off')
    
    # 4. Confidence distribution
    ax4 = plt.subplot(2, 3, 4)
    all_probs = attn_results["all_probs"]
    classes = [chr(ord('a') + i) for i in range(26)]
    colors = ['red' if i == attn_results["class_idx"] else 'blue' for i in range(26)]
    
    plt.bar(classes, all_probs, color=colors)
    plt.title('Confidence Distribution Across Classes')
    plt.xlabel('Character')
    plt.ylabel('Confidence')
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    
    # 5. Dot pattern visualization
    ax5 = plt.subplot(2, 3, 5)
    grid_pattern = np.array(attn_results["grid_pattern"]).reshape(3, 2)
    sns.heatmap(grid_pattern, annot=True, cmap='Blues', cbar=False, 
                xticklabels=['Left', 'Right'], 
                yticklabels=['Top', 'Middle', 'Bottom'])
    plt.title('Dot Pattern Matrix')
    
    # 6. Explanation text
    ax6 = plt.subplot(2, 3, 6)
    explanation = generate_detailed_explanation(attn_results, yolo_label, yolo_conf)
    plt.text(0.1, 0.5, explanation, fontsize=12, va='center', ha='left', wrap=True)
    plt.title('Explanation')
    plt.axis('off')
    
    plt.tight_layout()
    
    # Save the comprehensive visualization
    viz_path = os.path.join(output_path, "comprehensive_visualization.png")
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Also save individual components
    crop_path = os.path.join(output_path, "crop.png")
    cv2.imwrite(crop_path, crop_img)
    
    heatmap_path = os.path.join(output_path, "heatmap.png")
    cv2.imwrite(heatmap_path, overlay)
    
    grid_path = os.path.join(output_path, "grid.png")
    cv2.imwrite(grid_path, grid_img)
    
    return {
        "crop_path": crop_path,
        "heatmap_path": heatmap_path,
        "grid_path": grid_path,
        "comprehensive_viz_path": viz_path
    }

def process_all_crops_with_attention(attn_model, saved_items, output_dir, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Process all crops with AttentionCNN and save comprehensive results"""
    explain_dir = os.path.join(output_dir, "xai_explanations")
    os.makedirs(explain_dir, exist_ok=True)
    
    for idx, item in enumerate(saved_items):
        # Use the cropped image if available, otherwise load from path
        if "crop_img" in item:
            crop_img = item["crop_img"]
        else:
            crop_path = item.get("crop_path", "")
            if not os.path.exists(crop_path):
                continue
                
            crop_img = cv2.imread(crop_path)
            if crop_img is None:
                continue
                
        # Process with AttentionCNN
        attn_results = process_crop_with_attention(attn_model, crop_img, device)
        
        # Generate explanation
        yolo_label = item.get("yolo_label", "?")
        yolo_conf = item.get("yolo_conf", 0)
        explanation = generate_detailed_explanation(attn_results, yolo_label, yolo_conf)
        
        # Save comprehensive visualization
        char_explain_dir = os.path.join(explain_dir, f"{idx:04d}")
        viz_paths = create_comprehensive_visualization(crop_img, attn_results, char_explain_dir, yolo_label, yolo_conf)
        
        # Save metadata
        metadata = {
            "index": idx,
            "yolo_label": yolo_label,
            "yolo_confidence": float(yolo_conf),  # Convert to Python float
            "attn_label": attn_results["label"],
            "attn_confidence": float(attn_results["confidence"]),
            "grid_pattern": [int(x) for x in attn_results["grid_pattern"]],  # Convert to Python int
            "dot_count": int(attn_results["dot_count"]),
            "all_probabilities": {chr(ord('a') + i): float(attn_results["all_probs"][i]) for i in range(26)},
            "reason": explanation,
            "visualization_paths": viz_paths
        }
        
        metadata_path = os.path.join(char_explain_dir, "explanation.json")
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        # Update item with AttentionCNN results
        item["attn_label"] = attn_results["label"]
        item["attn_conf"] = attn_results["confidence"]
        item["attn_grid"] = attn_results["grid_pattern"]
        item["explain_path"] = char_explain_dir
    
    return saved_items

def update_csv_with_attention(items, csv_path):
    """Update CSV file with AttentionCNN results"""
    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        rows = list(reader)
    
    # Add AttentionCNN columns to header if needed
    if "attn_label" not in rows[0]:
        rows[0].extend(["attn_label", "attn_confidence", "attn_grid", "explain_path"])
    
    # Add AttentionCNN data to each row
    for i, row in enumerate(rows[1:], 1):
        if i-1 < len(items):
            item = items[i-1]
            row.extend([
                item.get("attn_label", ""),
                f"{item.get('attn_conf', 0):.4f}",
                str(item.get("attn_grid", [])),
                item.get("explain_path", "")
            ])
        else:
            row.extend(["", "", "", ""])
    
    # Write updated CSV
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)

def generate_sliding_proposals(img, window_size=(40, 60), step=(30, 45)):
    """
    Generate sliding window proposals across the entire image.
    Returns a list of proposal boxes in [x1, y1, x2, y2] format.
    """
    h, w = img.shape[:2]
    proposals = []
    cell_w, cell_h = window_size
    step_x, step_y = step
    
    for y in range(0, h, step_y):
        for x in range(0, w, step_x):
            x2 = min(x + cell_w, w)
            y2 = min(y + cell_h, h)
            # Only add proposals that are at least half the window size
            if (x2 - x) >= cell_w//2 and (y2 - y) >= cell_h//2:
                proposals.append([x, y, x2, y2])
    
    return proposals

def fuse_yolo_with_proposals(yolo_boxes, yolo_cls_ids, yolo_confs, proposals, names, iou_threshold=0.3):
    """
    Fuse YOLO detections with sliding window proposals.
    For each proposal, check if there's a YOLO detection with sufficient IoU.
    """
    fused_items = []
    
    # Convert YOLO boxes to a more accessible format
    yolo_detections = []
    for i, box in enumerate(yolo_boxes):
        yolo_detections.append({
            'box': box,
            'cls_id': yolo_cls_ids[i],
            'conf': yolo_confs[i],
            'label': names[yolo_cls_ids[i]] if names else str(yolo_cls_ids[i])
        })
    
    # For each proposal, find the best matching YOLO detection
    for proposal in proposals:
        best_iou = 0
        best_detection = None
        
        for det in yolo_detections:
            iou = calculate_iou(proposal, det['box'])
            if iou > best_iou:
                best_iou = iou
                best_detection = det
        
        # If we found a good match, use the YOLO detection
        if best_iou >= iou_threshold and best_detection:
            fused_items.append({
                "box": best_detection['box'].tolist() if hasattr(best_detection['box'], "tolist") else list(best_detection['box']),
                "label": best_detection['label'],
                "conf": float(best_detection['conf']),
                "method": "yolo_sliding_fused",
                "yolo_label": best_detection['label'],
                "yolo_conf": float(best_detection['conf'])
            })
        else:
            # Otherwise, mark as a potential space or unknown character
            fused_items.append({
                "box": proposal,
                "label": " ",  # Treat as space
                "conf": 0.1,   # Low confidence for proposal-based detection
                "method": "sliding_proposal",
                "yolo_label": " ",
                "yolo_conf": 0.1
            })
    
    return fused_items

def calculate_iou(box1, box2):
    """
    Calculate Intersection over Union (IoU) between two boxes.
    """
    # Determine coordinates of intersection rectangle
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # Calculate area of intersection
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    
    # Calculate area of both boxes
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    # Calculate union area
    union = area1 + area2 - intersection
    
    # Avoid division by zero
    if union == 0:
        return 0
    
    # Return IoU
    return intersection / union

def sliding_window_rpn_decoding(model, img, boxes, cls_ids, confs, names):
    """
    Combine YOLO detections with sliding window proposals for complete coverage.
    """
    # Generate sliding window proposals
    proposals = generate_sliding_proposals(img)
    
    # Fuse YOLO detections with proposals
    items = fuse_yolo_with_proposals(boxes, cls_ids, confs, proposals, names)
    
    return items

class EnhancedSpellChecker:
    def __init__(self, dictionary_path):
        self.word_freq = {}
        if dictionary_path and os.path.exists(dictionary_path):
            self.load_dictionary(dictionary_path)
    
    def load_dictionary(self, path):
        """Load frequency dictionary from file"""
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    word = parts[0].lower()
                    try:
                        freq = int(parts[1])
                        self.word_freq[word] = freq
                    except ValueError:
                        continue
    
    def contains(self, word):
        """Check if word exists in dictionary"""
        return word.lower() in self.word_freq
    
    def correct_sentence(self, sentence):
        words = sentence.lower().split()
        corrected_words = []
        
        for word in words:
            # Check if word is in dictionary
            if self.contains(word):
                corrected_words.append(word)
            else:
                # Try to split the word into dictionary words
                split_found = False
                for i in range(3, len(word)-2):
                    part1 = word[:i]
                    part2 = word[i:]
                    
                    if self.contains(part1) and self.contains(part2):
                        corrected_words.append(part1)
                        corrected_words.append(part2)
                        split_found = True
                        break
                
                if not split_found:
                    corrected_words.append(word)  # Keep original if no split found
        
        return " ".join(corrected_words)

def fix_line_boundaries(text: str) -> str:
    """Ensure spaces between words across line breaks."""
    # Replace newline without space → add space
    text = text.replace("\n", " ")
    # Collapse multiple spaces
    text = " ".join(text.split())
    return text

def advanced_dot_detection(crop_img):
    """
    Advanced dot detection using the CV2-based approach with preprocessing,
    grid line removal, and blob detection.
    """
    if crop_img is None or crop_img.size == 0:
        return []
    
    # Convert to grayscale if needed
    if len(crop_img.shape) == 3:
        gray = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)
    else:
        gray = crop_img
    
    # ===================== Preprocess =====================
    # Denoise
    img_blur = cv2.medianBlur(gray, 5)

    # Adaptive threshold
    thresh = cv2.adaptiveThreshold(
        img_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )

    # Ensure dots are white, background black
    white_ratio = np.sum(thresh == 255) / thresh.size
    if white_ratio > 0.5:
        thresh = cv2.bitwise_not(thresh)

    # ===================== Remove grid lines =====================
    # Detect vertical lines
    kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15))
    grid_v = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_v)

    # Detect horizontal lines
    kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
    grid_h = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_h)

    # Combine grid masks
    grid_lines = cv2.bitwise_or(grid_v, grid_h)

    # Subtract grids from threshold to keep only dots
    dots_only = cv2.subtract(thresh, grid_lines)

    # Cleanup small gaps
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    dots_only = cv2.morphologyEx(dots_only, cv2.MORPH_CLOSE, kernel)

    # ===================== Blob Detection =====================
    params = cv2.SimpleBlobDetector_Params()
    params.filterByArea = True
    params.minArea = 5
    params.maxArea = 1000
    params.filterByCircularity = True
    params.minCircularity = 0.4
    params.filterByConvexity = False
    params.filterByInertia = False

    detector = cv2.SimpleBlobDetector_create(params)
    keypoints = detector.detect(dots_only)

    # Fallback if nothing found
    if len(keypoints) == 0:
        print("⚠️ No blobs found → fallback to connected components")
        n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(dots_only, connectivity=8)
        keypoints = []
        for i in range(1, n_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            if 3 < area < 500:
                x, y = int(centroids[i][0]), int(centroids[i][1])
                size = int(np.sqrt(area))
                kp = cv2.KeyPoint(x, y, size)
                keypoints.append(kp)

    print(f"✅ Found {len(keypoints)} dot(s)")
    return keypoints

# Braille mapping (Grade-1 basic)
BRAILLE_TO_CHAR = {
    # index layout (1..6):
    # (1 4)
    # (2 5)
    # (3 6)
    (1,0,0,0,0,0): 'a',
    (1,1,0,0,0,0): 'b',
    (1,0,0,1,0,0): 'c',
    (1,0,0,1,1,0): 'd',
    (1,0,0,0,1,0): 'e',
    (1,1,0,1,0,0): 'f',
    (1,1,0,1,1,0): 'g',
    (1,1,0,0,1,0): 'h',
    (0,1,0,1,0,0): 'i',
    (0,1,0,1,1,0): 'j',
    (1,0,1,0,0,0): 'k',
    (1,1,1,0,0,0): 'l',
    (1,0,1,1,0,0): 'm',
    (1,0,1,1,1,0): 'n',
    (1,0,1,0,1,0): 'o',
    (1,1,1,1,0,0): 'p',
    (1,1,1,1,1,0): 'q',
    (1,1,1,0,1,0): 'r',
    (0,1,1,1,0,0): 's',
    (0,1,1,1,1,0): 't',
    (1,0,1,0,0,1): 'u',
    (1,1,1,0,0,1): 'v',
    (0,1,0,1,1,1): 'w',
    (1,0,1,1,0,1): 'x',
    (1,0,1,1,1,1): 'y',
    (1,0,1,0,1,1): 'z',
    (0,0,0,0,0,0): ' ',  # Space
}

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(NumpyEncoder, self).default(obj)

def sanitize_filename(char):
    if not char or str(char).isspace():
        return "space"
    if char == "?":
        return "unknown"
    safe = re.sub(r'[^A-Za-z0-9_-]', '', str(char))
    return safe or "unknown"

def write_csv(items, csv_path):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["index", "yolo_label", "yolo_confidence", "grid_label", "grid_confidence", 
                    "final_label", "x1", "y1", "x2", "y2", "padded_x1", "padded_y1", 
                    "padded_x2", "padded_y2", "crop_path", "grid_pattern"])
        for idx, it in enumerate(items):
            x1, y1, x2, y2 = [int(v) for v in it["box"]] if isinstance(it["box"], (list, np.ndarray)) else (0, 0, 0, 0)
            px1, py1, px2, py2 = it.get("padded_box", (x1, y1, x2, y2))
            w.writerow([
                idx, 
                it.get("yolo_label", ""), 
                f"{it.get('yolo_conf', 0):.4f}",
                it.get("grid_label", ""), 
                f"{it.get('grid_conf', 0):.4f}",
                it["label"],
                x1, y1, x2, y2, px1, py1, px2, py2,
                it.get("crop_path", ""), 
                str(it.get("grid", []))
            ])

def save_crops(img, items, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    counts = defaultdict(int)
    saved = []
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        w, h = x2 - x1, y2 - y1
        padx, pady = int(0.15 * w), int(0.15 * h)
        x1n = max(0, x1 - padx)
        y1n = max(0, y1 - pady)
        x2n = min(img.shape[1], x2 + padx)
        y2n = min(img.shape[0], y2 + pady)
        crop = img[y1n:y2n, x1n:x2n]
        if crop.size == 0:
            continue
        base = sanitize_filename(it["label"])
        counts[base] += 1
        fname = f"{base}-{counts[base]:03d}.png"
        fpath = os.path.join(out_dir, fname)
        cv2.imwrite(fpath, crop)
        it["crop_path"] = fpath
        it["padded_box"] = (x1n, y1n, x2n, y2n)
        saved.append(it)
    return saved

def log_metrics(image_path, total_chars, yolo_success, grid_success, final_success, 
                total_spaces, output_dir, method):
    os.makedirs(os.path.join(output_dir, 'logs'), exist_ok=True)
    log_path = os.path.join(output_dir, f"logs/detection_log_{datetime.now().strftime('%Y%m%d')}.csv")
    file_exists = os.path.exists(log_path)
    with open(log_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['Timestamp', 'Image', 'Method', 'Total', 'YOLO_Success', 
                            'Grid_Success', 'Final_Success', 'Spaces', 'YOLO_Rate', 'Grid_Rate', 'Final_Rate'])
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        yolo_rate = yolo_success / total_chars if total_chars > 0 else 0
        grid_rate = grid_success / total_chars if total_chars > 0 else 0
        final_rate = final_success / total_chars if total_chars > 0 else 0
        writer.writerow([
            timestamp, os.path.basename(image_path), method, total_chars, 
            yolo_success, grid_success, final_success, total_spaces,
            f"{yolo_rate:.2%}", f"{grid_rate:.2%}", f"{final_rate:.2%}"
        ])

def draw_grid_on_image(image, grid_pattern):
    h, w = image.shape[:2]
    grid_img = image.copy()
    # vertical midline
    cv2.line(grid_img, (w // 2, 0), (w // 2, h), (0, 255, 0), 1)
    # two horizontal lines (3 rows)
    for r in range(1, 3):
        cv2.line(grid_img, (0, r * h // 3), (w, r * h // 3), (0, 255, 0), 1)
    positions = [
        (int(w * 0.25), int(h * (1/6))),  # 1
        (int(w * 0.25), int(h * (3/6))),  # 2
        (int(w * 0.25), int(h * (5/6))),  # 3
        (int(w * 0.75), int(h * (1/6))),  # 4
        (int(w * 0.75), int(h * (3/6))),  # 5
        (int(w * 0.75), int(h * (5/6))),  # 6
    ]
    dot_radius = max(2, min(w, h) // 15)
    for i, has_dot in enumerate(grid_pattern):
        if has_dot:
            cv2.circle(grid_img, positions[i], dot_radius, (0, 0, 255), -1)
    return grid_img

def visualize_gaps(img, boxes, rows, avg_width, avg_height, total_spaces, word_gaps):
    debug_img = img.copy()
    for box in boxes:
        cv2.rectangle(debug_img, (int(box[0]), int(box[1])),
                      (int(box[2]), int(box[3])), (0, 255, 0), 2)
    for i, row in enumerate(rows):
        if i > 0:
            prev_bottom = max([b[3] for b in rows[i-1]])
            current_top = min([b[1] for b in row])
            gap = current_top - prev_bottom
            cv2.line(debug_img, (0, int(prev_bottom)),
                     (debug_img.shape[1], int(prev_bottom)), (255, 0, 0), 2)
            cv2.line(debug_img, (0, int(current_top)),
                     (debug_img.shape[1], int(current_top)), (0, 0, 255), 2)
            cv2.putText(debug_img, f"Row Gap: {gap:.1f}px",
                        (10, int((prev_bottom + current_top) / 2)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    
    # Visualize word gaps
    for gap_info in word_gaps:
        gap_center_x, gap_center_y, gap = gap_info
        cv2.line(debug_img, (int(gap_center_x - gap/2), int(gap_center_y)),
                 (int(gap_center_x + gap/2), int(gap_center_y)), (0, 255, 255), 2)
        cv2.putText(debug_img, f"{gap:.1f}px",
                    (int(gap_center_x - 20), int(gap_center_y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    cv2.putText(debug_img, f"Avg Width: {avg_width:.1f}px", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(debug_img, f"Avg Height: {avg_height:.1f}px", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(debug_img, f"Total Spaces: {total_spaces}", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    return debug_img

def cluster_rows(boxes, height_threshold=0.5):
    """Sort boxes into rows: top-to-bottom, left-to-right with improved clustering."""
    if len(boxes) == 0:
        return []
    
    # Handle single box case
    if len(boxes) == 1:
        return [boxes]
    
    # Calculate vertical centers
    vertical_centers = [(b[1] + b[3]) / 2.0 for b in boxes]
    avg_height = np.mean([b[3] - b[1] for b in boxes])
    
    # Use K-means clustering for better row separation
    vertical_centers = np.array(vertical_centers).reshape(-1, 1)
    
    # Try different numbers of clusters to find the best fit
    best_score = -np.inf
    best_clusters = None
    max_clusters = min(10, len(boxes))
    
    # Handle case where we have exactly 2 boxes
    if len(boxes) == 2:
        # Simple threshold-based clustering for 2 boxes
        if abs(vertical_centers[0] - vertical_centers[1]) < avg_height * height_threshold:
            return [boxes]  # Same row
        else:
            return [[boxes[0]], [boxes[1]]]  # Different rows
    
    # For 3+ boxes, use the original K-means approach
    for n_clusters in range(2, max_clusters + 1):
        if n_clusters > len(boxes):
            continue
            
        kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(vertical_centers)
        
        # Only calculate silhouette score if we have multiple clusters
        if n_clusters > 1:
            try:
                score = silhouette_score(vertical_centers, kmeans.labels_)
            except:
                score = -1  # Assign low score if silhouette fails
        else:
            score = -1
        
        if score > best_score:
            best_score = score
            best_clusters = kmeans.labels_
    
    # If no clusters found with n>=2, use single cluster
    if best_clusters is None:
        return [boxes]
    
    # Group boxes by cluster
    rows = [[] for _ in range(len(set(best_clusters)))]
    for i, cluster_id in enumerate(best_clusters):
        rows[cluster_id].append(boxes[i])
    
    # Sort rows by average y-position (top to bottom)
    row_centers = [np.mean([(b[1] + b[3]) / 2 for b in row]) for row in rows]
    sorted_indices = np.argsort(row_centers)
    sorted_rows = [rows[i] for i in sorted_indices]
    
    # Sort each row by x-position (left to right)
    for i, row in enumerate(sorted_rows):
        sorted_rows[i] = sorted(row, key=lambda b: b[0])
    
    return sorted_rows

def grid_from_rpn_bbox(char_img, keypoints):
    """Map detected dots to a 2x3 grid (col-major order)."""
    h, w = char_img.shape[:2]
    grid = np.zeros(6, dtype=int)
    if len(keypoints) == 0:
        return tuple(grid)
    cell_w = w / 2.0   # 2 columns
    cell_h = h / 3.0   # 3 rows
    for kp in keypoints:
        x, y = kp.pt
        col = min(1, max(0, int(x / cell_w)))   # 0 or 1
        row = min(2, max(0, int(y / cell_h)))   # 0,1,2
        pos = col * 3 + row   # col-major mapping -> 0..5
        grid[pos] = 1
    return tuple(grid)

def yolo_only_decoding(model, img, boxes, cls_ids, confs, names):
    items = []
    for i, box in enumerate(boxes):
        label = names[cls_ids[i]] if names else str(cls_ids[i])
        conf = float(confs[i])
        items.append({
            "box": box.tolist() if hasattr(box, "tolist") else list(box),
            "label": label,
            "conf": conf,
            "method": "yolo_only",
            "yolo_label": label,
            "yolo_conf": conf
        })
    return items

def rpn_grid_single(img, box, yolo_label, yolo_conf):
    """Apply RPN grid decoding to a single cropped detection"""
    x1, y1, x2, y2 = [int(v) for v in box]
    w, h = x2 - x1, y2 - y1
    padx, pady = int(0.15 * w), int(0.15 * h)
    x1n = max(0, x1 - padx)
    y1n = max(0, y1 - pady)
    x2n = min(img.shape[1], x2 + padx)
    y2n = min(img.shape[0], y2 + pady)
    crop_img = img[y1n:y2n, x1n:x2n]
    if crop_img.size == 0:
        return None

    # Use advanced dot detection
    keypoints = advanced_dot_detection(crop_img)
    grid_pattern = grid_from_rpn_bbox(crop_img, keypoints)
    
    # Map to character
    decoded = BRAILLE_TO_CHAR.get(grid_pattern, yolo_label)
    
    # Confidence based on dot detection quality
    grid_confidence = min(1.0, len(keypoints) / 6.0 + 0.3) if len(keypoints) > 0 else 0.5
    
    return {
        "box": box,
        "label": decoded,
        "conf": grid_confidence,
        "grid": grid_pattern,
        "method": "rpn_grid",
        "yolo_label": yolo_label,
        "yolo_conf": yolo_conf,
        "grid_label": decoded,
        "grid_conf": grid_confidence,
        "crop_img": crop_img   # store the cropped image for XAI
    }

def focused_rpn_grid_decoding(model, img, boxes, cls_ids, confs, names):
    """Focus on detected regions and apply RPN-based 2×3 grid analysis"""
    items = []
    yolo_items = yolo_only_decoding(model, img, boxes, cls_ids, confs, names)
    
    for yolo_item in yolo_items:
        grid_item = rpn_grid_single(img, yolo_item["box"], yolo_item["yolo_label"], yolo_item["yolo_conf"])
        
        if grid_item is None:
            items.append(yolo_item)
            continue
            
        # Choose the result with higher confidence
        if yolo_item["conf"] >= grid_item["conf"]:
            chosen_item = yolo_item
        else:
            chosen_item = grid_item
            
        items.append(chosen_item)
    
    return items

def assign_rows_to_items(items, rows):
    """
    Assign a row_id to each item by IoU overlap with row boxes.
    rows: list of list of boxes
    """
    def iou(a, b):
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b
        inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
        inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
        inter = iw * ih
        area_a = (ax2 - ax1) * (ay2 - ay1)
        area_b = (bx2 - bx1) * (by2 - by1)
        union = area_a + area_b - inter + 1e-6
        return inter / union

    for it in items:
        it["row_id"] = -1

    for row_id, row in enumerate(rows):
        for it in items:
            if it["row_id"] != -1:
                continue
            # choose the best overlap with any box in this row
            best = 0.0
            for rb in row:
                best = max(best, iou(it["box"], rb))
            if best > 0.2:  # overlap threshold
                it["row_id"] = row_id

    # any leftover -> assign by nearest row center in Y
    if rows:
        row_centers = [np.mean([(b[1]+b[3])/2.0 for b in r]) for r in rows]
        for it in items:
            if it["row_id"] == -1:
                cy = (it["box"][1] + it["box"][3]) / 2.0
                nearest = int(np.argmin([abs(cy - rc) for rc in row_centers]))
                it["row_id"] = nearest

    return items

def segment_words(row_items):
    """Adaptive word segmentation using K-means clustering on gaps"""
    if len(row_items) < 2:
        return [row_items]

    # Calculate gaps between characters
    gaps = []
    for i in range(len(row_items)-1):
        gap = row_items[i+1]["box"][0] - row_items[i]["box"][2]
        gaps.append(gap)
    
    # Handle case with only one gap (can't cluster with k=2)
    if len(gaps) == 1:
        # Use a simple threshold for single gap
        avg_char_width = np.mean([it["box"][2] - it["box"][0] for it in row_items])
        threshold = 1.5 * avg_char_width
        
        if gaps[0] > threshold:
            return [row_items[0:1], row_items[1:]]
        else:
            return [row_items]
    
    gaps = np.array(gaps).reshape(-1, 1)
    
    # Use K-means to find optimal gap threshold
    try:
        kmeans = KMeans(n_clusters=2, random_state=0, n_init=10).fit(gaps)
        labels = kmeans.labels_
        
        # Find which cluster represents larger gaps (word boundaries)
        if np.mean(gaps[labels == 0]) > np.mean(gaps[labels == 1]):
            word_boundary_label = 0
        else:
            word_boundary_label = 1
        
        threshold = np.mean(gaps[labels == word_boundary_label])
    except Exception as e:
        # Fallback if KMeans fails
        avg_gap = np.mean(gaps)
        threshold = avg_gap * 1.5  # Simple heuristic
    
    # Segment into words
    words = []
    current_word = [row_items[0]]
    
    for i, gap in enumerate(gaps.flatten()):
        if gap > threshold:
            words.append(current_word)
            current_word = [row_items[i+1]]
        else:
            current_word.append(row_items[i+1])
    words.append(current_word)
    
    return words

def detect_word_boundaries(items, rows, horizontal_space_threshold=1.5):
    """Improved word boundary detection using adaptive clustering"""
    # Group items by row
    grouped = defaultdict(list)
    for it in items:
        grouped[it["row_id"]].append(it)
    
    # Sort items in each row by x-coordinate
    for row_id in grouped:
        grouped[row_id] = sorted(grouped[row_id], key=lambda x: x["box"][0])
    
    # Calculate average character width
    if items:
        avg_width = np.mean([it["box"][2] - it["box"][0] for it in items])
    else:
        avg_width = 0
    
    # Process each row to detect word boundaries using adaptive clustering
    word_gaps = []
    for row_id, row_items in grouped.items():
        if len(row_items) < 2:
            continue
            
        # Use adaptive segmentation
        words = segment_words(row_items)
        
        # Mark word boundaries
        word_start_indices = [0]  # First word always starts at index 0
        for i in range(1, len(words)):
            # The first character of each subsequent word
            first_char_idx = sum(len(words[j]) for j in range(i))
            if first_char_idx < len(row_items):
                row_items[first_char_idx]["word_start"] = True
                word_start_indices.append(first_char_idx)
        
        # Calculate gaps for visualization
        for i in range(1, len(word_start_indices)):
            prev_idx = word_start_indices[i-1] + len(words[i-1]) - 1
            curr_idx = word_start_indices[i]
            
            if prev_idx < len(row_items) and curr_idx < len(row_items):
                prev_box = row_items[prev_idx]["box"]
                curr_box = row_items[curr_idx]["box"]
                
                gap = curr_box[0] - prev_box[2]
                gap_center_x = (prev_box[2] + curr_box[0]) / 2
                gap_center_y = (prev_box[1] + prev_box[3]) / 2
                
                word_gaps.append((gap_center_x, gap_center_y, gap))
    
    return items, word_gaps

def apply_spell_check(sentence, spell_checker):
    """Apply spell checking to fix common spacing and word issues"""
    if spell_checker is None:
        return sentence
    
    return spell_checker.correct_sentence(sentence)

def correct_with_transformer(sentence, method="byt5"):
    """Use character-level transformer for better correction of braille text"""
    try:
        if method == "byt5":
            # Use ByT5 for character-level correction (Seq2Seq model)
            tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
            model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")
            
            # Format input
            input_text = f"correct: {sentence}"
            inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
            
            # Generate correction
            with torch.no_grad():
                outputs = model.generate(
                    inputs.input_ids,
                    max_length=512,
                    num_beams=5,
                    early_stopping=True
                )
            
            corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
            return corrected
            
        elif method == "whitespace":
            # Use a specialized whitespace correction model
            try:
                corrector = pipeline("text2text-generation", 
                                    model="prithivida/whitespace_correction")
                corrected = corrector(sentence)
                return corrected[0]['generated_text']
            except:
                # Fallback if model not available
                print("Whitespace correction model not available, using simple method")
                # Simple whitespace correction as fallback
                words = sentence.split()
                return " ".join(words)
            
        else:
            # Fallback to grammar correction
            try:
                corrector = pipeline("text2text-generation", 
                                    model="pszemraj/flan-t5-large-grammar-synthesis")
                corrected = corrector(sentence)
                return corrected[0]['generated_text']
            except:
                print("Grammar correction model not available")
                return sentence
            
    except Exception as e:
        print(f"Transformer correction failed: {e}")
        return sentence

def create_yolo_visualization(img, items):
    """Create visualization with YOLO detections"""
    yolo_img = img.copy()
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        label = it.get("yolo_label", "?")
        conf = it.get("yolo_conf", 0)
        cv2.rectangle(yolo_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(yolo_img, f"{label} ({conf:.2f})",
                    (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return yolo_img

def create_grid_visualization(img, items):
    """Create visualization with grid method detections"""
    grid_img = img.copy()
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        label = it.get("grid_label", "?")
        conf = it.get("grid_conf", 0)
        cv2.rectangle(grid_img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(grid_img, f"{label} ({conf:.2f})",
                    (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return grid_img

def create_final_visualization(img, items):
    """Create visualization with final detections"""
    final_img = img.copy()
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        label = it["label"]
        conf = it["conf"]
        method = it["method"]
        color = (0, 255, 0) if method == "yolo_only" else (0, 0, 255)
        cv2.rectangle(final_img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(final_img, f"{label} ({conf:.2f})",
                    (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return final_img

def text_to_speech(sentence):
    """
    Convert text to speech and read it aloud.
    """
    try:
        engine = pyttsx3.init()
        engine.say(sentence)
        engine.runAndWait()
    except Exception as e:
        print(f"Text-to-speech failed: {e}")

def process_image_with_yolo_grid(model, image_path, output_dir, conf_threshold=0.05):
    """
    Process image using a grid-based approach to ensure full coverage
    """
    os.makedirs(output_dir, exist_ok=True)
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Error loading {image_path}")
        return None, []

    # Get image dimensions
    h, w = img.shape[:2]
    
    # Run YOLO prediction with low confidence threshold
    results = model.predict(
        img,
        conf=conf_threshold,  # Low threshold to capture weak detections
        iou=0.6,
        agnostic_nms=True,
        max_det=5000,
        verbose=False
    )
    r0 = results[0]

    # Collect detections
    boxes = r0.boxes.xyxy.cpu().numpy() if r0.boxes is not None else []
    cls_ids = r0.boxes.cls.cpu().numpy().astype(int) if r0.boxes is not None else []
    confs = r0.boxes.conf.cpu().numpy() if r0.boxes is not None else []
    names = r0.names if hasattr(r0, "names") else {}
    
    # Calculate average character size for grid estimation
    if len(boxes) > 0:
        widths = [b[2] - b[0] for b in boxes]
        heights = [b[3] - b[1] for b in boxes]
        avg_width = float(np.median(widths))
        avg_height = float(np.median(heights))
        cell_w = int(avg_width * 1.2)  # Add 20% padding
        cell_h = int(avg_height * 1.2)
    else:
        # Default cell size if no boxes detected
        avg_width = avg_height = 0.0
        cell_w, cell_h = 40, 60
    
    # Calculate grid dimensions
    cols = w // cell_w
    rows_count = h // cell_h
    
    # Initialize grid with empty values
    grid = {}
    for r in range(rows_count):
        for c in range(cols):
            grid[(r, c)] = (" ", 0.0, None)  # (label, confidence, box)

    # Map detections to grid cells
    for box, cls_id, conf in zip(boxes, cls_ids, confs):
        x1, y1, x2, y2 = box
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2
        
        # Determine grid cell
        c = int(center_x // cell_w)
        r = int(center_y // cell_h)
        
        # Ensure within bounds
        if 0 <= r < rows_count and 0 <= c < cols:
            label = names.get(cls_id, " ")
            # Keep only the highest confidence detection per cell
            if (r, c) not in grid or conf > grid[(r, c)][1]:
                grid[(r, c)] = (label, conf, box)

    # Reconstruct text row by row
    lines = []
    items = []
    for r in range(rows_count):
        line = ""
        for c in range(cols):
            label, conf, box = grid[(r, c)]
            line += label
            
            # Create item for visualization and logging
            if box is not None:
                items.append({
                    "box": box.tolist(),
                    "label": label,
                    "conf": conf,
                    "method": "grid_based",
                    "yolo_label": label,
                    "yolo_conf": conf,
                    "grid_label": label,
                    "grid_conf": conf
                })
            else:
                # Create a dummy box for empty cells
                dummy_box = [c*cell_w, r*cell_h, (c+1)*cell_w, (r+1)*cell_h]
                items.append({
                    "box": dummy_box,
                    "label": " ",
                    "conf": 0.0,
                    "method": "grid_based",
                    "yolo_label": " ",
                    "yolo_conf": 0.0,
                    "grid_label": " ",
                    "grid_conf": 0.0
                })
        lines.append(line)

    # Save reconstructed text
    txt_path = os.path.join(output_dir, "grid_based_detection.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    # Create visualization
    visualized_img = img.copy()
    for item in items:
        x1, y1, x2, y2 = [int(v) for v in item["box"]]
        label = item["label"]
        conf = item["conf"]
        
        # Draw rectangle
        color = (0, 255, 0) if label != " " else (0, 0, 255)
        cv2.rectangle(visualized_img, (x1, y1), (x2, y2), color, 2)
        
        # Draw label
        if label != " ":
            cv2.putText(visualized_img, f"{label} ({conf:.2f})", 
                       (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Draw grid lines
    for r in range(rows_count + 1):
        y = r * cell_h
        cv2.line(visualized_img, (0, y), (w, y), (255, 0, 0), 1)
    for c in range(cols + 1):
        x = c * cell_w
        cv2.line(visualized_img, (x, 0), (x, h), (255, 0, 0), 1)

    # Save visualization
    vis_path = os.path.join(output_dir, "grid_based_visualization.png")
    cv2.imwrite(vis_path, visualized_img)

    # Save CSV with detections
    csv_path = os.path.join(output_dir, "grid_based_detections.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["row", "col", "label", "confidence", "x1", "y1", "x2", "y2"])
        for r in range(rows_count):
            for c in range(cols):
                label, conf, box = grid[(r, c)]
                if box is not None:
                    x1, y1, x2, y2 = box
                else:
                    x1, y1, x2, y2 = c*cell_w, r*cell_h, (c+1)*cell_w, (r+1)*cell_h
                writer.writerow([r, c, label, conf, x1, y1, x2, y2])

    print(f"✅ Grid-based processing complete")
    print(f"✅ Saved text to {txt_path}")
    print(f"✅ Saved visualization to {vis_path}")
    print(f"✅ Saved detections to {csv_path}")
    
    return "\n".join(lines), items

def create_sentence_level_xai(final_sentence, items, output_dir):
    """Create sentence-level XAI report"""
    # Count agreements between YOLO and AttentionCNN
    agreements = 0
    disagreements = 0
    total_chars = 0
    
    for item in items:
        if "attn_label" in item and "yolo_label" in item:
            total_chars += 1
            if item["attn_label"].lower() == item["yolo_label"].lower():
                agreements += 1
            else:
                disagreements += 1
    
    agreement_rate = agreements / total_chars if total_chars > 0 else 0
    
    # Create explanation
    if agreement_rate > 0.8:
        reason = "High agreement between YOLO detections and AttentionCNN interpretations. The sentence is likely accurate."
    elif agreement_rate > 0.6:
        reason = "Moderate agreement between detection methods. Some characters may need verification."
    else:
        reason = "Low agreement between detection methods. Manual verification recommended."
    
    explanation = {
        "sentence": final_sentence,
        "total_characters": total_chars,
        "agreements": agreements,
        "disagreements": disagreements,
        "agreement_rate": agreement_rate,
        "reason": reason,
        "timestamp": datetime.now().isoformat()
    }
    
    # Create visualization
    plt.figure(figsize=(10, 6))
    labels = ['Agreements', 'Disagreements']
    values = [agreements, disagreements]
    colors = ['green', 'red']
    
    plt.bar(labels, values, color=colors)
    plt.title('YOLO vs AttentionCNN Agreement')
    plt.ylabel('Count')
    plt.text(0, agreements + 0.1, f'{agreements}', ha='center')
    plt.text(1, disagreements + 0.1, f'{disagreements}', ha='center')
    
    # Save visualization
    viz_path = os.path.join(output_dir, "sentence_level_agreement.png")
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save explanation
    explain_path = os.path.join(output_dir, "sentence_explanation.json")
    with open(explain_path, 'w') as f:
        json.dump(explanation, f, indent=2, cls=NumpyEncoder)
    
    return explain_path, viz_path

def process_braille_image(
    model,
    image_path=None,
    method="yolo_only",
    horizontal_space_threshold=1.5,
    vertical_space_threshold=0.8,
    newline_threshold=1.5,
    save_outputs=True,
    enable_spell_check=True,
    enable_transformer=True,
    enable_tts=True,
    dictionary_path=None,
    transformer_method="byt5",
    attention_model_path=None
):
    # Load AttentionCNN model if provided
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    attn_model = None
    if attention_model_path and os.path.exists(attention_model_path):
        try:
            attn_model = load_attention_model(attention_model_path, device)
            print(f"✅ Loaded AttentionCNN model from {attention_model_path}")
        except Exception as e:
            print(f"❌ Failed to load AttentionCNN model: {e}")
            attn_model = None
    
    # Initialize spell checker if enabled
    spell_checker = EnhancedSpellChecker(dictionary_path) if enable_spell_check and dictionary_path else None
    
    # Handle grid-based method
    if method == "grid_based":
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_dir = f"grid_based_detection_{ts}"
        os.makedirs(out_dir, exist_ok=True)
        
        # Load image for additional processing
        img = cv2.imread(image_path)
        if img is None:
            print(f"❌ Error loading {image_path}")
            return None, [], 0
        
        # Run grid-based processing
        final_sentence, items = process_image_with_yolo_grid(
            model, image_path, out_dir
        )
        
        # Apply spell checking if enabled
        if enable_spell_check:
            corrected_sentence = apply_spell_check(final_sentence, spell_checker)
        else:
            corrected_sentence = final_sentence

        # Apply transformer correction if enabled
        if enable_transformer:
            # First do whitespace correction, then grammar correction if needed
            whitespace_corrected = correct_with_transformer(corrected_sentence, method="whitespace")
            if transformer_method == "byt5":
                transformer_corrected = correct_with_transformer(whitespace_corrected, method="byt5")
            else:
                transformer_corrected = correct_with_transformer(whitespace_corrected, method="grammar")
            
            print(f"TRANSFORMER CORRECTED SENTENCE:\n{transformer_corrected}")
            final_sentence = transformer_corrected
        else:
            final_sentence = corrected_sentence

        # Text-to-speech if enabled
        if enable_tts:
            text_to_speech(final_sentence)
            
        # ADDITIONAL SAVING FUNCTIONALITY FOR GRID_BASED METHOD
        if save_outputs:
            # Save crops
            saved_items = save_crops(img, items, os.path.join(out_dir, "crops"))
            
            # Process with AttentionCNN if model is available
            if attn_model is not None:
                saved_items = process_all_crops_with_attention(attn_model, saved_items, out_dir, device)
                
                # Create sentence-level XAI
                create_sentence_level_xai(final_sentence, saved_items, out_dir)
            
            # Write detailed CSV
            write_csv(saved_items, os.path.join(out_dir, "detailed_mapping.csv"))
            
            # Update CSV with AttentionCNN results if available
            if attn_model is not None:
                update_csv_with_attention(saved_items, os.path.join(out_dir, "detailed_mapping.csv"))
            
            # Create visualizations
            yolo_img = create_yolo_visualization(img, items)
            grid_img = create_grid_visualization(img, items)
            final_img = create_final_visualization(img, items)
            
            # For gap visualization, we need to extract boxes from items
            boxes = [item["box"] for item in items if "box" in item]
            if boxes:
                rows = cluster_rows(boxes)
                # Calculate average dimensions for visualization
                widths = [b[2] - b[0] for b in boxes]
                heights = [b[3] - b[1] for b in boxes]
                avg_width = float(np.median(widths)) if widths else 0
                avg_height = float(np.median(heights)) if heights else 0
                
                # Detect word boundaries
                items_with_rows = assign_rows_to_items(items, rows)
                items_with_words, word_gaps = detect_word_boundaries(items_with_rows, rows, horizontal_space_threshold)
                
                gap_img = visualize_gaps(img, boxes, rows, avg_width, avg_height, 0, word_gaps)
                cv2.imwrite(os.path.join(out_dir, "gap_visualization.png"), gap_img)
            
            cv2.imwrite(os.path.join(out_dir, "yolo_detections.png"), yolo_img)
            cv2.imwrite(os.path.join(out_dir, "grid_detections.png"), grid_img)
            cv2.imwrite(os.path.join(out_dir, "final_detections.png"), final_img)

            # Save final results with all processing information
            with open(os.path.join(out_dir, "final_result.txt"), "w", encoding="utf-8") as f:
                f.write("Grid-based Detection:\n")
                f.write(final_sentence)
                f.write("\n\nPost-Processing Applied:\n")
                f.write(f"Spell Checking: {enable_spell_check}\n")
                f.write(f"Transformer Correction: {enable_transformer}\n")
                f.write(f"Text-to-Speech: {enable_tts}\n")
                if attn_model is not None:
                    f.write(f"AttentionCNN XAI: Enabled\n")
                
            print(f"\nGrid-based processing complete. All results saved to: {out_dir}")
            print(f"FINAL SENTENCE:\n{final_sentence}")
            
            # Calculate success rates for logging
            yolo_success = sum(1 for it in items if it.get("yolo_label", "?") != '?' and it.get("yolo_label", "?") != ' ')
            grid_success = sum(1 for it in items if it.get("grid_label", "?") != '?' and it.get("grid_label", "?") != ' ')
            final_success = sum(1 for it in items if it["label"] != '?' and it["label"] != ' ')
            
            log_metrics(image_path, len(items), yolo_success, grid_success, final_success, 
                        0, out_dir, method)  # 0 spaces as grid_based doesn't calculate them
        
        return final_sentence, items, 0  # spaces count is not applicable for grid-based

    # For other methods, use the original processing
    img = cv2.imread(image_path)
    if img is None:
        print(f"Image not found: {image_path}")
        return "", []
    
    max_dim = min(max(img.shape[:2]), 1920)
    try:
        stride = int(model.stride.max().item())
    except Exception:
        stride = 32
    imgsz = int(np.ceil(max_dim / stride) * stride)

    results = model.predict(img, imgsz=imgsz, verbose=False)
    r0 = results[0]

    boxes = r0.boxes.xyxy.cpu().numpy()
    cls_ids = r0.boxes.cls.cpu().numpy().astype(int)
    confs = r0.boxes.conf.cpu().numpy() if hasattr(r0.boxes, "conf") else np.ones(len(cls_ids))
    names = model.names if hasattr(model, "names") else None

    # Sort boxes by position (left to right, top to bottom)
    if len(boxes) > 0:
        # Calculate center points
        centers = [(box[0] + box[2]) / 2 for box in boxes], [(box[1] + box[3]) / 2 for box in boxes]
        
        # Sort by y-position first (rows), then by x-position (within rows)
        sorted_indices = np.lexsort((centers[0], centers[1]))
        boxes = boxes[sorted_indices]
        cls_ids = cls_ids[sorted_indices]
        confs = confs[sorted_indices] if hasattr(confs, '__getitem__') else confs

    if len(boxes) > 0:
        widths = [b[2] - b[0] for b in boxes]
        heights = [b[3] - b[1] for b in boxes]
        avg_width = float(np.median(widths))
        avg_height = float(np.median(heights))
    else:
        avg_width = avg_height = 0.0

    if method == "yolo_only":
        items = yolo_only_decoding(model, img, boxes, cls_ids, confs, names)
    elif method == "rpn_grid":
        items = rpn_grid_decoding(model, img, boxes, cls_ids, confs, names)
    elif method == "sliding_window_rpn":
        items = sliding_window_rpn_decoding(model, img, boxes, cls_ids, confs, names)
    elif method == "focused_rpn_grid":
        items = focused_rpn_grid_decoding(model, img, boxes, cls_ids, confs, names)
    else:
        raise ValueError("Method must be 'yolo_only', 'rpn_grid', 'grid_based', 'sliding_window_rpn', or 'focused_rpn_grid'")

    # Cluster rows from raw boxes and assign to items
    rows = cluster_rows(list(boxes))
    assign_rows_to_items(items, rows)
    
    # Detect word boundaries
    items, word_gaps = detect_word_boundaries(items, rows, horizontal_space_threshold)

    # Group by row_id and assemble sentence with adaptive spacing
    grouped = defaultdict(list)
    for it in items:
        grouped[it["row_id"]].append(it)

    sentence_parts = []
    total_spaces = 0

    for row_id in sorted(grouped.keys()):
        row_items = sorted(grouped[row_id], key=lambda it: it["box"][0])
        row_text = []
        for i, it in enumerate(row_items):
            if i > 0:
                prev_box = row_items[i-1]["box"]
                box = it["box"]
                gap = box[0] - prev_box[2]
                char_width = ((prev_box[2] - prev_box[0]) + (box[2] - box[0])) / 2.0
                adaptive_threshold = horizontal_space_threshold * char_width
                if gap > adaptive_threshold or it.get("word_start", False):
                    row_text.append(" ")
                    total_spaces += 1
            row_text.append(it["label"])
        row_str = "".join(row_text)

        if row_id > 0:
            current_top = min([it["box"][1] for it in row_items])
            prev_bottom = max([it["box"][3] for it in grouped[row_id - 1]])
            gap_rows = current_top - prev_bottom
            line_gap_threshold = vertical_space_threshold * avg_height
            newline_gap_threshold = newline_threshold * avg_height
            if gap_rows > newline_gap_threshold:
                sentence_parts.append("\n")
            elif gap_rows > line_gap_threshold:
                sentence_parts.append(" ")
                total_spaces += 1

        sentence_parts.append(row_str)

    raw_sentence = "".join(sentence_parts)
    
    # Fix line boundaries (ONEWHO\nLOSTHIS → ONE WHO LOST HIS)
    raw_sentence = fix_line_boundaries(raw_sentence)
    
    # Apply spell checking if enabled
    if enable_spell_check:
        corrected_sentence = apply_spell_check(raw_sentence, spell_checker)
    else:
        corrected_sentence = raw_sentence

    # Apply transformer correction if enabled
    if enable_transformer:
        # First do whitespace correction, then grammar correction if needed
        whitespace_corrected = correct_with_transformer(corrected_sentence, method="whitespace")
        if transformer_method == "byt5":
            transformer_corrected = correct_with_transformer(whitespace_corrected, method="byt5")
        else:
            transformer_corrected = correct_with_transformer(whitespace_corrected, method="grammar")
        
        print(f"TRANSFORMER CORRECTED SENTENCE:\n{transformer_corrected}")
        final_sentence = transformer_corrected
    else:
        final_sentence = corrected_sentence

    # Text-to-speech if enabled
    if enable_tts:
        text_to_speech(final_sentence)

    # Create visualizations
    yolo_img = create_yolo_visualization(img, items)
    grid_img = create_grid_visualization(img, items)
    final_img = create_final_visualization(img, items)
    gap_img = visualize_gaps(img, boxes, rows, avg_width, avg_height, total_spaces, word_gaps)

    if save_outputs:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_dir = f"detected_characters_{method}_{ts}"
        os.makedirs(out_dir, exist_ok=True)

        saved_items = save_crops(img, items, os.path.join(out_dir, "crops"))
        
        # Process with AttentionCNN if model is available
        if attn_model is not None:
            saved_items = process_all_crops_with_attention(attn_model, saved_items, out_dir, device)
            
            # Create sentence-level XAI
            create_sentence_level_xai(final_sentence, saved_items, out_dir)
        
        write_csv(saved_items, os.path.join(out_dir, "mapping.csv"))
        
        # Update CSV with AttentionCNN results if available
        if attn_model is not None:
            update_csv_with_attention(saved_items, os.path.join(out_dir, "mapping.csv"))
        
        cv2.imwrite(os.path.join(out_dir, "yolo_detections.png"), yolo_img)
        cv2.imwrite(os.path.join(out_dir, "grid_detections.png"), grid_img)
        cv2.imwrite(os.path.join(out_dir, "final_detections.png"), final_img)
        cv2.imwrite(os.path.join(out_dir, "gap_visualization.png"), gap_img)

        with open(os.path.join(out_dir, "decoded_sentence.txt"), "w", encoding="utf-8") as f:
            f.write("Raw Sentence:\n")
            f.write(raw_sentence)
            f.write("\n\nCorrected Sentence:\n")
            f.write(corrected_sentence)
            if enable_transformer:
                f.write("\n\nTransformer Corrected Sentence:\n")
                f.write(transformer_corrected)
                f.write("\n\nFinal Sentence:\n")
                f.write(final_sentence)

        if method == "rpn_grid" or method == "focused_rpn_grid":
            grid_dir = os.path.join(out_dir, "grid_visualizations")
            os.makedirs(grid_dir, exist_ok=True)
            for it in saved_items:
                if "crop_path" in it and "grid" in it:
                    char_img = cv2.imread(it["crop_path"])
                    if char_img is not None:
                        grid_img = draw_grid_on_image(char_img, it["grid"])
                        base_name = os.path.splitext(os.path.basename(it["crop_path"]))[0]
                        grid_path = os.path.join(grid_dir, f"{base_name}_grid.png")
                        cv2.imwrite(grid_path, grid_img)

        # Calculate success rates
        yolo_success = sum(1 for it in items if it.get("yolo_label", "?") != '?')
        grid_success = sum(1 for it in items if it.get("grid_label", "?") != '?')
        final_success = sum(1 for it in items if it["label"] != '?')
        
        log_metrics(image_path, len(items), yolo_success, grid_success, final_success, 
                    total_spaces, out_dir, method)
        print(f"\nDebugging files saved to: {out_dir}")

    # Show windows
    cv2.imshow("YOLO Detections", yolo_img)
    cv2.imshow("Grid Detections", grid_img)
    cv2.imshow("Final Detections", final_img)
    cv2.imshow("Gap Visualization", gap_img)
    
    print("\n" + "="*50)
    print(f"RAW SENTENCE:\n{raw_sentence}")
    print(f"CORRECTED SENTENCE:\n{corrected_sentence}")
    if enable_transformer:
        print(f"TRANSFORMER CORRECTED SENTENCE:\n{transformer_corrected}")
    print(f"FINAL SENTENCE:\n{final_sentence}")
    print(f"TOTAL SPACES: {total_spaces}")
    print("="*50)
    
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    return final_sentence, items, total_spaces

# --------------------------- Run ---------------------------
if __name__ == "__main__":
    # Update these paths
    model_path = r"D:\Papers\Braille\braille_char\v8n\weights\best.pt"
    image_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-15 001800.png"
    dictionary_path = r"C:\Users\mogno\Downloads\frequency_dictionary_en_82_765.txt"
    attention_model_path = r"C:\Users\mogno\Downloads\braille_best_model.pth"
    
    model = YOLO(model_path)

    # Choose: "yolo_only", "rpn_grid", "grid_based", "sliding_window_rpn", or "focused_rpn_grid"
    mode = "focused_rpn_grid"

    if mode == "yolo_only":
        process_braille_image(
            model,
            image_path=image_path,
            method="yolo_only",
            save_outputs=True,
            enable_spell_check=True,
            enable_transformer=True,
            enable_tts=True,
            dictionary_path=dictionary_path,
            transformer_method="byt5",
            attention_model_path=attention_model_path
        )
    elif mode == "rpn_grid":
        process_braille_image(
            model,
            image_path=image_path,
            method="rpn_grid",
            save_outputs=True,
            enable_spell_check=True,
            enable_transformer=True,
            enable_tts=True,
            dictionary_path=dictionary_path,
            transformer_method="byt5",
            attention_model_path=attention_model_path
        )
    elif mode == "grid_based":
        process_braille_image(
            model,
            image_path=image_path,
            method="grid_based",
            save_outputs=True,
            enable_spell_check=True,
            enable_transformer=True,
            enable_tts=True,
            dictionary_path=dictionary_path,
            transformer_method="byt5",
            attention_model_path=attention_model_path
        )
    elif mode == "sliding_window_rpn":
        process_braille_image(
            model,
            image_path=image_path,
            method="sliding_window_rpn",
            save_outputs=True,
            enable_spell_check=True,
            enable_transformer=True,
            enable_tts=True,
            dictionary_path=dictionary_path,
            transformer_method="byt5",
            attention_model_path=attention_model_path
        )
    elif mode == "focused_rpn_grid":
        process_braille_image(
            model,
            image_path=image_path,
            method="focused_rpn_grid",
            save_outputs=True,
            enable_spell_check=True,
            enable_transformer=True,
            enable_tts=True,
            dictionary_path=dictionary_path,
            transformer_method="byt5",
            attention_model_path=attention_model_path
        )

✅ Found 2 dot(s)
✅ Found 3 dot(s)
✅ Found 2 dot(s)
✅ Found 2 dot(s)
✅ Found 1 dot(s)
✅ Found 3 dot(s)
✅ Found 3 dot(s)
✅ Found 3 dot(s)
✅ Found 4 dot(s)
✅ Found 4 dot(s)
✅ Found 3 dot(s)
✅ Found 3 dot(s)
✅ Found 2 dot(s)
✅ Found 3 dot(s)
✅ Found 3 dot(s)
✅ Found 3 dot(s)
✅ Found 4 dot(s)
✅ Found 4 dot(s)
✅ Found 2 dot(s)
✅ Found 3 dot(s)
✅ Found 3 dot(s)
✅ Found 1 dot(s)
⚠️ No blobs found → fallback to connected components
✅ Found 0 dot(s)
✅ Found 3 dot(s)
✅ Found 4 dot(s)
✅ Found 2 dot(s)
✅ Found 4 dot(s)
✅ Found 3 dot(s)
✅ Found 2 dot(s)
✅ Found 3 dot(s)
✅ Found 3 dot(s)
✅ Found 4 dot(s)
✅ Found 4 dot(s)
✅ Found 4 dot(s)
✅ Found 1 dot(s)
✅ Found 2 dot(s)
✅ Found 1 dot(s)
✅ Found 5 dot(s)
✅ Found 4 dot(s)
✅ Found 2 dot(s)
✅ Found 2 dot(s)
✅ Found 2 dot(s)
✅ Found 3 dot(s)
✅ Found 3 dot(s)
✅ Found 4 dot(s)
✅ Found 2 dot(s)
✅ Found 1 dot(s)
✅ Found 3 dot(s)
✅ Found 1 dot(s)
✅ Found 1 dot(s)
✅ Found 3 dot(s)
✅ Found 4 dot(s)
✅ Found 3 dot(s)
✅ Found 4 dot(s)
✅ Found 3 dot(s)
✅ Found 2 do

**Ablation**

In [4]:
import os
import re
import csv
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datetime import datetime
from ultralytics import YOLO
from collections import defaultdict
import pyttsx3
import json
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns
from matplotlib import cm
from tqdm import tqdm

# clustering helpers
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Transformer imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

class DotGridHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, 1, 1)

    def forward(self, feat):
        hmap = self.conv(feat)  # Bx1xhxw
        grid_logits = F.adaptive_avg_pool2d(hmap, (3, 2))  # Bx1x3x2
        return hmap, grid_logits

class ClassFromGridHead(nn.Module):
    def __init__(self, num_classes=26):
        super().__init__()
        self.fc = nn.Linear(6, num_classes)

    def forward(self, grid_probs):
        x = grid_probs.view(grid_probs.size(0), -1)
        return self.fc(x)

class BrailleXAI(nn.Module):
    def __init__(self, num_classes=26):
        super().__init__()
        # Adjusted for 96x64 input images
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
        )
        self.dot_head = DotGridHead(128)
        self.class_from_grid = ClassFromGridHead(num_classes)

    def forward(self, x):
        feat = self.backbone(x)  # Bx128x24x16 (for 96x64 input)
        hmap, grid_logits = self.dot_head(feat)
        grid_probs = torch.sigmoid(grid_logits)
        class_logits = self.class_from_grid(grid_probs)
        return {
            "class_logits": class_logits,
            "grid_logits": grid_logits,
            "grid_probs": grid_probs,
            "heatmap": hmap
        }

def load_attention_model(model_path, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Load the trained BrailleXAI model"""
    model = BrailleXAI(num_classes=26)  # 26 letters
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

def process_crop_with_attention(model, crop_img, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Process a single crop with the BrailleXAI model"""
    # Convert to grayscale if needed
    if len(crop_img.shape) == 3:
        gray = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)
    else:
        gray = crop_img
    
    # Resize to expected input size (64x96 as used in training)
    resized = cv2.resize(gray, (64, 96))  # Width, Height
    
    # Normalize and convert to tensor
    tensor_img = torch.from_numpy(resized).float() / 255.0
    tensor_img = tensor_img.unsqueeze(0).unsqueeze(0)  # Add batch and channel dims
    tensor_img = tensor_img.to(device)
    
    # Run through model
    with torch.no_grad():
        outputs = model(tensor_img)
    
    # Process outputs
    class_probs = F.softmax(outputs["class_logits"], dim=1)
    class_conf, class_idx = torch.max(class_probs, dim=1)
    
    # Map class index to character (0-25: a-z)
    attn_label = chr(ord('a') + class_idx.item())
    
    # Get grid pattern
    grid_pattern = (outputs["grid_probs"] > 0.5).int().squeeze().cpu().numpy().flatten().tolist()
    
    # Get heatmap
    heatmap = outputs["heatmap"].squeeze().cpu().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = cv2.resize(heatmap, (crop_img.shape[1], crop_img.shape[0]))
    heatmap = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # Calculate dot count from grid pattern
    dot_count = sum(grid_pattern)
    
    # Get class probabilities for all classes
    all_probs = class_probs.squeeze().cpu().numpy()
    
    return {
        "label": attn_label,
        "confidence": class_conf.item(),
        "grid_pattern": grid_pattern,
        "heatmap": heatmap,
        "dot_count": dot_count,
        "all_probs": all_probs,
        "class_idx": class_idx.item()
    }

def generate_detailed_explanation(attn_results, yolo_label, yolo_conf):
    """Generate detailed explanation text based on attention results"""
    grid = attn_results["grid_pattern"]
    label = attn_results["label"]
    conf = attn_results["confidence"]
    dot_count = attn_results["dot_count"]
    
    # Describe the grid pattern
    dot_positions = []
    positions = ["top-left", "middle-left", "bottom-left", "top-right", "middle-right", "bottom-right"]
    
    for i, has_dot in enumerate(grid):
        if has_dot:
            dot_positions.append(positions[i])
    
    # Start building the explanation
    explanation_parts = []
    
    # Basic detection info
    explanation_parts.append(f"Character detected as '{label}' with {conf:.1%} confidence.")
    
    # Dot pattern explanation
    if dot_positions:
        dots_desc = ", ".join(dot_positions)
        explanation_parts.append(f"Detected dots at: {dots_desc}.")
        explanation_parts.append(f"Total dots: {dot_count}.")
    else:
        explanation_parts.append("No dots detected.")
    
    # Compare with YOLO prediction
    if yolo_label.lower() != label and yolo_label != "?":
        explanation_parts.append(f"Disagrees with YOLO prediction '{yolo_label}' (confidence: {yolo_conf:.1%}).")
        
        # Provide possible reasons for disagreement
        if dot_count == 0 and yolo_label != " ":
            explanation_parts.append("Possible reason: YOLO detected a character but no dots were found.")
        elif dot_count > 0 and yolo_label == " ":
            explanation_parts.append("Possible reason: Dots were detected but YOLO predicted a space.")
        else:
            explanation_parts.append("Possible reason: Different interpretation of dot pattern.")
    else:
        explanation_parts.append(f"Confirms YOLO prediction '{yolo_label}' (confidence: {yolo_conf:.1%}).")
    
    # Add confidence assessment
    if conf > 0.9:
        explanation_parts.append("High confidence in this prediction.")
    elif conf > 0.7:
        explanation_parts.append("Moderate confidence in this prediction.")
    else:
        explanation_parts.append("Low confidence in this prediction. Consider manual verification.")
    
    return " ".join(explanation_parts)

def create_comprehensive_visualization(crop_img, attn_results, output_path, yolo_label, yolo_conf):
    """Create comprehensive visualization for a single character"""
    os.makedirs(output_path, exist_ok=True)
    
    # Create a figure with multiple subplots
    fig = plt.figure(figsize=(20, 12))
    
    # 1. Original crop
    ax1 = plt.subplot(2, 3, 1)
    if len(crop_img.shape) == 3:
        plt.imshow(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(crop_img, cmap='gray')
    plt.title(f'Original Crop\nYOLO: {yolo_label} ({yolo_conf:.1%})')
    plt.axis('off')
    
    # 2. Heatmap overlay
    ax2 = plt.subplot(2, 3, 2)
    heatmap = attn_results["heatmap"]
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(crop_img, 0.6, heatmap_color, 0.4, 0)
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title('Attention Heatmap Overlay')
    plt.axis('off')
    
    # 3. Grid visualization
    ax3 = plt.subplot(2, 3, 3)
    grid_img = draw_grid_on_image(crop_img, attn_results["grid_pattern"])
    plt.imshow(cv2.cvtColor(grid_img, cv2.COLOR_BGR2RGB))
    plt.title('Detected Dot Pattern')
    plt.axis('off')
    
    # 4. Confidence distribution
    ax4 = plt.subplot(2, 3, 4)
    all_probs = attn_results["all_probs"]
    classes = [chr(ord('a') + i) for i in range(26)]
    colors = ['red' if i == attn_results["class_idx"] else 'blue' for i in range(26)]
    
    plt.bar(classes, all_probs, color=colors)
    plt.title('Confidence Distribution Across Classes')
    plt.xlabel('Character')
    plt.ylabel('Confidence')
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    
    # 5. Dot pattern visualization
    ax5 = plt.subplot(2, 3, 5)
    grid_pattern = np.array(attn_results["grid_pattern"]).reshape(3, 2)
    sns.heatmap(grid_pattern, annot=True, cmap='Blues', cbar=False, 
                xticklabels=['Left', 'Right'], 
                yticklabels=['Top', 'Middle', 'Bottom'])
    plt.title('Dot Pattern Matrix')
    
    # 6. Explanation text
    ax6 = plt.subplot(2, 3, 6)
    explanation = generate_detailed_explanation(attn_results, yolo_label, yolo_conf)
    plt.text(0.1, 0.5, explanation, fontsize=12, va='center', ha='left', wrap=True)
    plt.title('Explanation')
    plt.axis('off')
    
    plt.tight_layout()
    
    # Save the comprehensive visualization
    viz_path = os.path.join(output_path, "comprehensive_visualization.png")
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Also save individual components
    crop_path = os.path.join(output_path, "crop.png")
    cv2.imwrite(crop_path, crop_img)
    
    heatmap_path = os.path.join(output_path, "heatmap.png")
    cv2.imwrite(heatmap_path, overlay)
    
    grid_path = os.path.join(output_path, "grid.png")
    cv2.imwrite(grid_path, grid_img)
    
    return {
        "crop_path": crop_path,
        "heatmap_path": heatmap_path,
        "grid_path": grid_path,
        "comprehensive_viz_path": viz_path
    }

def process_all_crops_with_attention(attn_model, saved_items, output_dir, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Process all crops with AttentionCNN and save comprehensive results"""
    explain_dir = os.path.join(output_dir, "xai_explanations")
    os.makedirs(explain_dir, exist_ok=True)
    
    for idx, item in tqdm(enumerate(saved_items), desc="Processing crops with AttentionCNN", total=len(saved_items)):
        # Use the cropped image if available, otherwise load from path
        if "crop_img" in item:
            crop_img = item["crop_img"]
        else:
            crop_path = item.get("crop_path", "")
            if not os.path.exists(crop_path):
                continue
                
            crop_img = cv2.imread(crop_path)
            if crop_img is None:
                continue
                
        # Process with AttentionCNN
        attn_results = process_crop_with_attention(attn_model, crop_img, device)
        
        # Generate explanation
        yolo_label = item.get("yolo_label", "?")
        yolo_conf = item.get("yolo_conf", 0)
        explanation = generate_detailed_explanation(attn_results, yolo_label, yolo_conf)
        
        # Save comprehensive visualization
        char_explain_dir = os.path.join(explain_dir, f"{idx:04d}")
        viz_paths = create_comprehensive_visualization(crop_img, attn_results, char_explain_dir, yolo_label, yolo_conf)
        
        # Save metadata
        metadata = {
            "index": idx,
            "yolo_label": yolo_label,
            "yolo_confidence": float(yolo_conf),  # Convert to Python float
            "attn_label": attn_results["label"],
            "attn_confidence": float(attn_results["confidence"]),
            "grid_pattern": [int(x) for x in attn_results["grid_pattern"]],  # Convert to Python int
            "dot_count": int(attn_results["dot_count"]),
            "all_probabilities": {chr(ord('a') + i): float(attn_results["all_probs"][i]) for i in range(26)},
            "reason": explanation,
            "visualization_paths": viz_paths
        }
        
        metadata_path = os.path.join(char_explain_dir, "explanation.json")
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        # Update item with AttentionCNN results
        item["attn_label"] = attn_results["label"]
        item["attn_conf"] = attn_results["confidence"]
        item["attn_grid"] = attn_results["grid_pattern"]
        item["explain_path"] = char_explain_dir
    
    return saved_items

def update_csv_with_attention(items, csv_path):
    """Update CSV file with AttentionCNN results"""
    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        rows = list(reader)
    
    # Add AttentionCNN columns to header if needed
    if "attn_label" not in rows[0]:
        rows[0].extend(["attn_label", "attn_confidence", "attn_grid", "explain_path"])
    
    # Add AttentionCNN data to each row
    for i, row in enumerate(rows[1:], 1):
        if i-1 < len(items):
            item = items[i-1]
            row.extend([
                item.get("attn_label", ""),
                f"{item.get('attn_conf', 0):.4f}",
                str(item.get("attn_grid", [])),
                item.get("explain_path", "")
            ])
        else:
            row.extend(["", "", "", ""])
    
    # Write updated CSV
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)

def generate_sliding_proposals(img, window_size=(40, 60), step=(30, 45)):
    """
    Generate sliding window proposals across the entire image.
    Returns a list of proposal boxes in [x1, y1, x2, y2] format.
    """
    h, w = img.shape[:2]
    proposals = []
    cell_w, cell_h = window_size
    step_x, step_y = step
    
    for y in range(0, h, step_y):
        for x in range(0, w, step_x):
            x2 = min(x + cell_w, w)
            y2 = min(y + cell_h, h)
            # Only add proposals that are at least half the window size
            if (x2 - x) >= cell_w//2 and (y2 - y) >= cell_h//2:
                proposals.append([x, y, x2, y2])
    
    return proposals

def fuse_yolo_with_proposals(yolo_boxes, yolo_cls_ids, yolo_confs, proposals, names, iou_threshold=0.3):
    """
    Fuse YOLO detections with sliding window proposals.
    For each proposal, check if there's a YOLO detection with sufficient IoU.
    """
    fused_items = []
    
    # Convert YOLO boxes to a more accessible format
    yolo_detections = []
    for i, box in enumerate(yolo_boxes):
        yolo_detections.append({
            'box': box,
            'cls_id': yolo_cls_ids[i],
            'conf': yolo_confs[i],
            'label': names[yolo_cls_ids[i]] if names else str(yolo_cls_ids[i])
        })
    
    # For each proposal, find the best matching YOLO detection
    for proposal in proposals:
        best_iou = 0
        best_detection = None
        
        for det in yolo_detections:
            iou = calculate_iou(proposal, det['box'])
            if iou > best_iou:
                best_iou = iou
                best_detection = det
        
        # If we found a good match, use the YOLO detection
        if best_iou >= iou_threshold and best_detection:
            fused_items.append({
                "box": best_detection['box'].tolist() if hasattr(best_detection['box'], "tolist") else list(best_detection['box']),
                "label": best_detection['label'],
                "conf": float(best_detection['conf']),
                "method": "yolo_sliding_fused",
                "yolo_label": best_detection['label'],
                "yolo_conf": float(best_detection['conf'])
            })
        else:
            # Otherwise, mark as a potential space or unknown character
            fused_items.append({
                "box": proposal,
                "label": " ",  # Treat as space
                "conf": 0.1,   # Low confidence for proposal-based detection
                "method": "sliding_proposal",
                "yolo_label": " ",
                "yolo_conf": 0.1
            })
    
    return fused_items

def calculate_iou(box1, box2):
    """
    Calculate Intersection over Union (IoU) between two boxes.
    """
    # Determine coordinates of intersection rectangle
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # Calculate area of intersection
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    
    # Calculate area of both boxes
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    # Calculate union area
    union = area1 + area2 - intersection
    
    # Avoid division by zero
    if union == 0:
        return 0
    
    # Return IoU
    return intersection / union

def sliding_window_rpn_decoding(model, img, boxes, cls_ids, confs, names):
    """
    Combine YOLO detections with sliding window proposals for complete coverage.
    """
    # Generate sliding window proposals
    proposals = generate_sliding_proposals(img)
    
    # Fuse YOLO detections with proposals
    items = fuse_yolo_with_proposals(boxes, cls_ids, confs, proposals, names)
    
    return items

class EnhancedSpellChecker:
    def __init__(self, dictionary_path):
        self.word_freq = {}
        if dictionary_path and os.path.exists(dictionary_path):
            self.load_dictionary(dictionary_path)
    
    def load_dictionary(self, path):
        """Load frequency dictionary from file"""
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    word = parts[0].lower()
                    try:
                        freq = int(parts[1])
                        self.word_freq[word] = freq
                    except ValueError:
                        continue
    
    def contains(self, word):
        """Check if word exists in dictionary"""
        return word.lower() in self.word_freq
    
    def correct_sentence(self, sentence):
        words = sentence.lower().split()
        corrected_words = []
        
        for word in words:
            # Check if word is in dictionary
            if self.contains(word):
                corrected_words.append(word)
            else:
                # Try to split the word into dictionary words
                split_found = False
                for i in range(3, len(word)-2):
                    part1 = word[:i]
                    part2 = word[i:]
                    
                    if self.contains(part1) and self.contains(part2):
                        corrected_words.append(part1)
                        corrected_words.append(part2)
                        split_found = True
                        break
                
                if not split_found:
                    corrected_words.append(word)  # Keep original if no split found
        
        return " ".join(corrected_words)

def fix_line_boundaries(text: str) -> str:
    """Ensure spaces between words across line breaks."""
    # Replace newline without space → add space
    text = text.replace("\n", " ")
    # Collapse multiple spaces
    text = " ".join(text.split())
    return text

def advanced_dot_detection(crop_img):
    """
    Advanced dot detection using the CV2-based approach with preprocessing,
    grid line removal, and blob detection.
    """
    if crop_img is None or crop_img.size == 0:
        return []
    
    # Convert to grayscale if needed
    if len(crop_img.shape) == 3:
        gray = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)
    else:
        gray = crop_img
    
    # ===================== Preprocess =====================
    # Denoise
    img_blur = cv2.medianBlur(gray, 5)

    # Adaptive threshold
    thresh = cv2.adaptiveThreshold(
        img_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )

    # Ensure dots are white, background black
    white_ratio = np.sum(thresh == 255) / thresh.size
    if white_ratio > 0.5:
        thresh = cv2.bitwise_not(thresh)

    # ===================== Remove grid lines =====================
    # Detect vertical lines
    kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15))
    grid_v = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_v)

    # Detect horizontal lines
    kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
    grid_h = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_h)

    # Combine grid masks
    grid_lines = cv2.bitwise_or(grid_v, grid_h)

    # Subtract grids from threshold to keep only dots
    dots_only = cv2.subtract(thresh, grid_lines)

    # Cleanup small gaps
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    dots_only = cv2.morphologyEx(dots_only, cv2.MORPH_CLOSE, kernel)

    # ===================== Blob Detection =====================
    params = cv2.SimpleBlobDetector_Params()
    params.filterByArea = True
    params.minArea = 5
    params.maxArea = 1000
    params.filterByCircularity = True
    params.minCircularity = 0.4
    params.filterByConvexity = False
    params.filterByInertia = False

    detector = cv2.SimpleBlobDetector_create(params)
    keypoints = detector.detect(dots_only)

    # Fallback if nothing found
    if len(keypoints) == 0:
        n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(dots_only, connectivity=8)
        keypoints = []
        for i in range(1, n_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            if 3 < area < 500:
                x, y = int(centroids[i][0]), int(centroids[i][1])
                size = int(np.sqrt(area))
                kp = cv2.KeyPoint(x, y, size)
                keypoints.append(kp)

    return keypoints

# Braille mapping (Grade-1 basic)
BRAILLE_TO_CHAR = {
    # index layout (1..6):
    # (1 4)
    # (2 5)
    # (3 6)
    (1,0,0,0,0,0): 'a',
    (1,1,0,0,0,0): 'b',
    (1,0,0,1,0,0): 'c',
    (1,0,0,1,1,0): 'd',
    (1,0,0,0,1,0): 'e',
    (1,1,0,1,0,0): 'f',
    (1,1,0,1,1,0): 'g',
    (1,1,0,0,1,0): 'h',
    (0,1,0,1,0,0): 'i',
    (0,1,0,1,1,0): 'j',
    (1,0,1,0,0,0): 'k',
    (1,1,1,0,0,0): 'l',
    (1,0,1,1,0,0): 'm',
    (1,0,1,1,1,0): 'n',
    (1,0,1,0,1,0): 'o',
    (1,1,1,1,0,0): 'p',
    (1,1,1,1,1,0): 'q',
    (1,1,1,0,1,0): 'r',
    (0,1,1,1,0,0): 's',
    (0,1,1,1,1,0): 't',
    (1,0,1,0,0,1): 'u',
    (1,1,1,0,0,1): 'v',
    (0,1,0,1,1,1): 'w',
    (1,0,1,1,0,1): 'x',
    (1,0,1,1,1,1): 'y',
    (1,0,1,0,1,1): 'z',
    (0,0,0,0,0,0): ' ',  # Space
}

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(NumpyEncoder, self).default(obj)

def sanitize_filename(char):
    if not char or str(char).isspace():
        return "space"
    if char == "?":
        return "unknown"
    safe = re.sub(r'[^A-Za-z0-9_-]', '', str(char))
    return safe or "unknown"

def write_csv(items, csv_path):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["index", "yolo_label", "yolo_confidence", "grid_label", "grid_confidence", 
                    "final_label", "x1", "y1", "x2", "y2", "padded_x1", "padded_y1", 
                    "padded_x2", "padded_y2", "crop_path", "grid_pattern"])
        for idx, it in enumerate(items):
            x1, y1, x2, y2 = [int(v) for v in it["box"]] if isinstance(it["box"], (list, np.ndarray)) else (0, 0, 0, 0)
            px1, py1, px2, py2 = it.get("padded_box", (x1, y1, x2, y2))
            w.writerow([
                idx, 
                it.get("yolo_label", ""), 
                f"{it.get('yolo_conf', 0):.4f}",
                it.get("grid_label", ""), 
                f"{it.get('grid_conf', 0):.4f}",
                it["label"],
                x1, y1, x2, y2, px1, py1, px2, py2,
                it.get("crop_path", ""), 
                str(it.get("grid", []))
            ])

def save_crops(img, items, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    counts = defaultdict(int)
    saved = []
    for it in tqdm(items, desc="Saving crops", total=len(items)):
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        w, h = x2 - x1, y2 - y1
        padx, pady = int(0.15 * w), int(0.15 * h)
        x1n = max(0, x1 - padx)
        y1n = max(0, y1 - pady)
        x2n = min(img.shape[1], x2 + padx)
        y2n = min(img.shape[0], y2 + pady)
        crop = img[y1n:y2n, x1n:x2n]
        if crop.size == 0:
            continue
        base = sanitize_filename(it["label"])
        counts[base] += 1
        fname = f"{base}-{counts[base]:03d}.png"
        fpath = os.path.join(out_dir, fname)
        cv2.imwrite(fpath, crop)
        it["crop_path"] = fpath
        it["padded_box"] = (x1n, y1n, x2n, y2n)
        saved.append(it)
    return saved

def log_metrics(image_path, total_chars, yolo_success, grid_success, final_success, 
                total_spaces, output_dir, method):
    os.makedirs(os.path.join(output_dir, 'logs'), exist_ok=True)
    log_path = os.path.join(output_dir, f"logs/detection_log_{datetime.now().strftime('%Y%m%d')}.csv")
    file_exists = os.path.exists(log_path)
    with open(log_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['Timestamp', 'Image', 'Method', 'Total', 'YOLO_Success', 
                            'Grid_Success', 'Final_Success', 'Spaces', 'YOLO_Rate', 'Grid_Rate', 'Final_Rate'])
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        yolo_rate = yolo_success / total_chars if total_chars > 0 else 0
        grid_rate = grid_success / total_chars if total_chars > 0 else 0
        final_rate = final_success / total_chars if total_chars > 0 else 0
        writer.writerow([
            timestamp, os.path.basename(image_path), method, total_chars, 
            yolo_success, grid_success, final_success, total_spaces,
            f"{yolo_rate:.2%}", f"{grid_rate:.2%}", f"{final_rate:.2%}"
        ])

def draw_grid_on_image(image, grid_pattern):
    h, w = image.shape[:2]
    grid_img = image.copy()
    # vertical midline
    cv2.line(grid_img, (w // 2, 0), (w // 2, h), (0, 255, 0), 1)
    # two horizontal lines (3 rows)
    for r in range(1, 3):
        cv2.line(grid_img, (0, r * h // 3), (w, r * h // 3), (0, 255, 0), 1)
    positions = [
        (int(w * 0.25), int(h * (1/6))),  # 1
        (int(w * 0.25), int(h * (3/6))),  # 2
        (int(w * 0.25), int(h * (5/6))),  # 3
        (int(w * 0.75), int(h * (1/6))),  # 4
        (int(w * 0.75), int(h * (3/6))),  # 5
        (int(w * 0.75), int(h * (5/6))),  # 6
    ]
    dot_radius = max(2, min(w, h) // 15)
    for i, has_dot in enumerate(grid_pattern):
        if has_dot:
            cv2.circle(grid_img, positions[i], dot_radius, (0, 0, 255), -1)
    return grid_img

def visualize_gaps(img, boxes, rows, avg_width, avg_height, total_spaces, word_gaps):
    debug_img = img.copy()
    for box in boxes:
        cv2.rectangle(debug_img, (int(box[0]), int(box[1])),
                      (int(box[2]), int(box[3])), (0, 255, 0), 2)
    for i, row in enumerate(rows):
        if i > 0:
            prev_bottom = max([b[3] for b in rows[i-1]])
            current_top = min([b[1] for b in row])
            gap = current_top - prev_bottom
            cv2.line(debug_img, (0, int(prev_bottom)),
                     (debug_img.shape[1], int(prev_bottom)), (255, 0, 0), 2)
            cv2.line(debug_img, (0, int(current_top)),
                     (debug_img.shape[1], int(current_top)), (0, 0, 255), 2)
            cv2.putText(debug_img, f"Row Gap: {gap:.1f}px",
                        (10, int((prev_bottom + current_top) / 2)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    
    # Visualize word gaps
    for gap_info in word_gaps:
        gap_center_x, gap_center_y, gap = gap_info
        cv2.line(debug_img, (int(gap_center_x - gap/2), int(gap_center_y)),
                 (int(gap_center_x + gap/2), int(gap_center_y)), (0, 255, 255), 2)
        cv2.putText(debug_img, f"{gap:.1f}px",
                    (int(gap_center_x - 20), int(gap_center_y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    cv2.putText(debug_img, f"Avg Width: {avg_width:.1f}px", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(debug_img, f"Avg Height: {avg_height:.1f}px", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(debug_img, f"Total Spaces: {total_spaces}", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    return debug_img

def cluster_rows(boxes, height_threshold=0.5):
    """Sort boxes into rows: top-to-bottom, left-to-right with improved clustering."""
    if len(boxes) == 0:
        return []
    
    # Handle single detection case
    if len(boxes) == 1:
        return [boxes]
    
    # Calculate vertical centers
    vertical_centers = [(b[1] + b[3]) / 2.0 for b in boxes]
    avg_height = np.mean([b[3] - b[1] for b in boxes])
    
    # Use K-means clustering for better row separation
    vertical_centers = np.array(vertical_centers).reshape(-1, 1)
    
    # Try different numbers of clusters to find the best fit
    best_score = -np.inf
    best_clusters = None
    max_clusters = min(10, len(boxes))
    
    for n_clusters in range(2, max_clusters + 1):  # Start from 2 clusters
        if n_clusters > len(boxes):
            continue
            
        kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(vertical_centers)
        
        # Only calculate silhouette score if we have multiple clusters
        if n_clusters > 1:
            try:
                score = silhouette_score(vertical_centers, kmeans.labels_)
            except:
                score = -1  # Assign low score if silhouette fails
        else:
            score = -1
        
        if score > best_score:
            best_score = score
            best_clusters = kmeans.labels_
    
    # If no clusters found with n>=2, use single cluster
    if best_clusters is None:
        return [boxes]
    
    # Group boxes by cluster
    rows = [[] for _ in range(len(set(best_clusters)))]
    for i, cluster_id in enumerate(best_clusters):
        rows[cluster_id].append(boxes[i])
    
    # Sort rows by average y-position (top to bottom)
    row_centers = [np.mean([(b[1] + b[3]) / 2 for b in row]) for row in rows]
    sorted_indices = np.argsort(row_centers)
    sorted_rows = [rows[i] for i in sorted_indices]
    
    # Sort each row by x-position (left to right)
    for i, row in enumerate(sorted_rows):
        sorted_rows[i] = sorted(row, key=lambda b: b[0])
    
    return sorted_rows

def grid_from_rpn_bbox(char_img, keypoints):
    """Map detected dots to a 2x3 grid (col-major order)."""
    h, w = char_img.shape[:2]
    grid = np.zeros(6, dtype=int)
    if len(keypoints) == 0:
        return tuple(grid)
    cell_w = w / 2.0   # 2 columns
    cell_h = h / 3.0   # 3 rows
    for kp in keypoints:
        x, y = kp.pt
        col = min(1, max(0, int(x / cell_w)))   # 0 or 1
        row = min(2, max(0, int(y / cell_h)))   # 0,1,2
        pos = col * 3 + row   # col-major mapping -> 0..5
        grid[pos] = 1
    return tuple(grid)

def yolo_only_decoding(model, img, boxes, cls_ids, confs, names):
    items = []
    for i, box in enumerate(boxes):
        label = names[cls_ids[i]] if names else str(cls_ids[i])
        conf = float(confs[i])
        items.append({
            "box": box.tolist() if hasattr(box, "tolist") else list(box),
            "label": label,
            "conf": conf,
            "method": "yolo_only",
            "yolo_label": label,
            "yolo_conf": conf
        })
    return items

def rpn_grid_single(img, box, yolo_label, yolo_conf):
    """Apply RPN grid decoding to a single cropped detection"""
    x1, y1, x2, y2 = [int(v) for v in box]
    w, h = x2 - x1, y2 - y1
    padx, pady = int(0.15 * w), int(0.15 * h)
    x1n = max(0, x1 - padx)
    y1n = max(0, y1 - pady)
    x2n = min(img.shape[1], x2 + padx)
    y2n = min(img.shape[0], y2 + pady)
    crop_img = img[y1n:y2n, x1n:x2n]
    if crop_img.size == 0:
        return None

    # Use advanced dot detection
    keypoints = advanced_dot_detection(crop_img)
    grid_pattern = grid_from_rpn_bbox(crop_img, keypoints)
    
    # Map to character
    decoded = BRAILLE_TO_CHAR.get(grid_pattern, yolo_label)
    
    # Confidence based on dot detection quality
    grid_confidence = min(1.0, len(keypoints) / 6.0 + 0.3) if len(keypoints) > 0 else 0.5
    
    return {
        "box": box,
        "label": decoded,
        "conf": grid_confidence,
        "grid": grid_pattern,
        "method": "rpn_grid",
        "yolo_label": yolo_label,
        "yolo_conf": yolo_conf,
        "grid_label": decoded,
        "grid_conf": grid_confidence,
        "crop_img": crop_img   # store the cropped image for XAI
    }

def focused_rpn_grid_decoding(model, img, boxes, cls_ids, confs, names):
    """Focus on detected regions and apply RPN-based 2×3 grid analysis"""
    items = []
    yolo_items = yolo_only_decoding(model, img, boxes, cls_ids, confs, names)
    
    for yolo_item in tqdm(yolo_items, desc="Processing RPN grid", total=len(yolo_items)):
        grid_item = rpn_grid_single(img, yolo_item["box"], yolo_item["yolo_label"], yolo_item["yolo_conf"])
        
        if grid_item is None:
            items.append(yolo_item)
            continue
            
        # Choose the result with higher confidence
        if yolo_item["conf"] >= grid_item["conf"]:
            chosen_item = yolo_item
        else:
            chosen_item = grid_item
            
        items.append(chosen_item)
    
    return items

def assign_rows_to_items(items, rows):
    """
    Assign a row_id to each item by IoU overlap with row boxes.
    rows: list of list of boxes
    """
    def iou(a, b):
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b
        inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
        inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
        inter = iw * ih
        area_a = (ax2 - ax1) * (ay2 - ay1)
        area_b = (bx2 - bx1) * (by2 - by1)
        union = area_a + area_b - inter + 1e-6
        return inter / union

    for it in items:
        it["row_id"] = -1

    for row_id, row in enumerate(rows):
        for it in items:
            if it["row_id"] != -1:
                continue
            # choose the best overlap with any box in this row
            best = 0.0
            for rb in row:
                best = max(best, iou(it["box"], rb))
            if best > 0.2:  # overlap threshold
                it["row_id"] = row_id

    # any leftover -> assign by nearest row center in Y
    if rows:
        row_centers = [np.mean([(b[1]+b[3])/2.0 for b in r]) for r in rows]
        for it in items:
            if it["row_id"] == -1:
                cy = (it["box"][1] + it["box"][3]) / 2.0
                nearest = int(np.argmin([abs(cy - rc) for rc in row_centers]))
                it["row_id"] = nearest

    return items

def segment_words(row_items):
    """Adaptive word segmentation using K-means clustering on gaps"""
    if len(row_items) < 2:
        return [row_items]

    # Calculate gaps between characters
    gaps = []
    for i in range(len(row_items)-1):
        gap = row_items[i+1]["box"][0] - row_items[i]["box"][2]
        gaps.append(gap)
    
    # Handle case with only one gap (can't cluster with k=2)
    if len(gaps) == 1:
        # Use a simple threshold for single gap
        avg_char_width = np.mean([it["box"][2] - it["box"][0] for it in row_items])
        threshold = 1.5 * avg_char_width
        
        if gaps[0] > threshold:
            return [row_items[0:1], row_items[1:]]
        else:
            return [row_items]
    
    gaps = np.array(gaps).reshape(-1, 1)
    
    # Use K-means to find optimal gap threshold
    try:
        kmeans = KMeans(n_clusters=2, random_state=0, n_init=10).fit(gaps)
        labels = kmeans.labels_
        
        # Find which cluster represents larger gaps (word boundaries)
        if np.mean(gaps[labels == 0]) > np.mean(gaps[labels == 1]):
            word_boundary_label = 0
        else:
            word_boundary_label = 1
        
        threshold = np.mean(gaps[labels == word_boundary_label])
    except Exception as e:
        # Fallback if KMeans fails
        avg_gap = np.mean(gaps)
        threshold = avg_gap * 1.5  # Simple heuristic
    
    # Segment into words
    words = []
    current_word = [row_items[0]]
    
    for i, gap in enumerate(gaps.flatten()):
        if gap > threshold:
            words.append(current_word)
            current_word = [row_items[i+1]]
        else:
            current_word.append(row_items[i+1])
    words.append(current_word)
    
    return words

def apply_spell_check(sentence, spell_checker):
    """Apply spell checking to fix common spacing and word issues"""
    if spell_checker is None:
        return sentence
    
    return spell_checker.correct_sentence(sentence)

def correct_with_transformer(sentence, method="byt5"):
    """Use character-level transformer for better correction of braille text"""
    try:
        if method == "byt5":
            # Use ByT5 for character-level correction (Seq2Seq model)
            tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
            model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")
            
            # Format input
            input_text = f"correct: {sentence}"
            inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
            
            # Generate correction
            with torch.no_grad():
                outputs = model.generate(
                    inputs.input_ids,
                    max_length=512,
                    num_beams=5,
                    early_stopping=True
                )
            
            corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
            return corrected
            
        elif method == "whitespace":
            # Use a specialized whitespace correction model
            try:
                corrector = pipeline("text2text-generation", 
                                    model="prithivida/whitespace_correction")
                corrected = corrector(sentence)
                return corrected[0]['generated_text']
            except:
                # Fallback if model not available
                # Simple whitespace correction as fallback
                words = sentence.split()
                return " ".join(words)
            
        else:
            # Fallback to grammar correction
            try:
                corrector = pipeline("text2text-generation", 
                                    model="pszemraj/flan-t5-large-grammar-synthesis")
                corrected = corrector(sentence)
                return corrected[0]['generated_text']
            except:
                return sentence
            
    except Exception as e:
        return sentence

def create_yolo_visualization(img, items):
    """Create visualization with YOLO detections"""
    yolo_img = img.copy()
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        label = it.get("yolo_label", "?")
        conf = it.get("yolo_conf", 0)
        cv2.rectangle(yolo_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(yolo_img, f"{label} ({conf:.2f})",
                    (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return yolo_img

def create_grid_visualization(img, items):
    """Create visualization with grid method detections"""
    grid_img = img.copy()
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        label = it.get("grid_label", "?")
        conf = it.get("grid_conf", 0)
        cv2.rectangle(grid_img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(grid_img, f"{label} ({conf:.2f})",
                    (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return grid_img

def create_final_visualization(img, items):
    """Create visualization with final detections"""
    final_img = img.copy()
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        label = it["label"]
        conf = it["conf"]
        method = it["method"]
        color = (0, 255, 0) if method == "yolo_only" else (0, 0, 255)
        cv2.rectangle(final_img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(final_img, f"{label} ({conf:.2f})",
                    (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return final_img

def text_to_speech(sentence):
    """
    Convert text to speech and read it aloud.
    """
    try:
        engine = pyttsx3.init()
        engine.say(sentence)
        engine.runAndWait()
    except Exception as e:
        pass

def process_image_with_yolo_grid(model, image_path, output_dir, conf_threshold=0.05):
    """
    Process image using a grid-based approach to ensure full coverage
    """
    os.makedirs(output_dir, exist_ok=True)
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Error loading {image_path}")
        return None, []

    # Get image dimensions
    h, w = img.shape[:2]
    
    # Run YOLO prediction with low confidence threshold
    results = model.predict(
        img,
        conf=conf_threshold,  # Low threshold to capture weak detections
        iou=0.6,
        agnostic_nms=True,
        max_det=5000,
        verbose=False
    )
    r0 = results[0]

    # Collect detections
    boxes = r0.boxes.xyxy.cpu().numpy() if r0.boxes is not None else []
    cls_ids = r0.boxes.cls.cpu().numpy().astype(int) if r0.boxes is not None else []
    confs = r0.boxes.conf.cpu().numpy() if r0.boxes is not None else []
    names = r0.names if hasattr(r0, "names") else {}
    
    # Calculate average character size for grid estimation
    if len(boxes) > 0:
        widths = [b[2] - b[0] for b in boxes]
        heights = [b[3] - b[1] for b in boxes]
        avg_width = float(np.median(widths))
        avg_height = float(np.median(heights))
        cell_w = int(avg_width * 1.2)  # Add 20% padding
        cell_h = int(avg_height * 1.2)
    else:
        # Default cell size if no boxes detected
        avg_width = avg_height = 0.0
        cell_w, cell_h = 40, 60
    
    # Calculate grid dimensions
    cols = w // cell_w
    rows_count = h // cell_h
    
    # Initialize grid with empty values
    grid = {}
    for r in range(rows_count):
        for c in range(cols):
            grid[(r, c)] = (" ", 0.0, None)  # (label, confidence, box)

    # Map detections to grid cells
    for box, cls_id, conf in zip(boxes, cls_ids, confs):
        x1, y1, x2, y2 = box
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2
        
        # Determine grid cell
        c = int(center_x // cell_w)
        r = int(center_y // cell_h)
        
        # Ensure within bounds
        if 0 <= r < rows_count and 0 <= c < cols:
            label = names.get(cls_id, " ")
            # Keep only the highest confidence detection per cell
            if (r, c) not in grid or conf > grid[(r, c)][1]:
                grid[(r, c)] = (label, conf, box)

    # Reconstruct text row by row
    lines = []
    items = []
    for r in range(rows_count):
        line = ""
        for c in range(cols):
            label, conf, box = grid[(r, c)]
            line += label
            
            # Create item for visualization and logging
            if box is not None:
                items.append({
                    "box": box.tolist(),
                    "label": label,
                    "conf": conf,
                    "method": "grid_based",
                    "yolo_label": label,
                    "yolo_conf": conf,
                    "grid_label": label,
                    "grid_conf": conf
                })
            else:
                # Create a dummy box for empty cells
                dummy_box = [c*cell_w, r*cell_h, (c+1)*cell_w, (r+1)*cell_h]
                items.append({
                    "box": dummy_box,
                    "label": " ",
                    "conf": 0.0,
                    "method": "grid_based",
                    "yolo_label": " ",
                    "yolo_conf": 0.0,
                    "grid_label": " ",
                    "grid_conf": 0.0
                })
        lines.append(line)

    # Save reconstructed text
    txt_path = os.path.join(output_dir, "grid_based_detection.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    # Create visualization
    visualized_img = img.copy()
    for item in items:
        x1, y1, x2, y2 = [int(v) for v in item["box"]]
        label = item["label"]
        conf = item["conf"]
        
        # Draw rectangle
        color = (0, 255, 0) if label != " " else (0, 0, 255)
        cv2.rectangle(visualized_img, (x1, y1), (x2, y2), color, 2)
        
        # Draw label
        if label != " ":
            cv2.putText(visualized_img, f"{label} ({conf:.2f})", 
                       (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Draw grid lines
    for r in range(rows_count + 1):
        y = r * cell_h
        cv2.line(visualized_img, (0, y), (w, y), (255, 0, 0), 1)
    for c in range(cols + 1):
        x = c * cell_w
        cv2.line(visualized_img, (x, 0), (x, h), (255, 0, 0), 1)

    # Save visualization
    vis_path = os.path.join(output_dir, "grid_based_visualization.png")
    cv2.imwrite(vis_path, visualized_img)

    # Save CSV with detections
    csv_path = os.path.join(output_dir, "grid_based_detections.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["row", "col", "label", "confidence", "x1", "y1", "x2", "y2"])
        for r in range(rows_count):
            for c in range(cols):
                label, conf, box = grid[(r, c)]
                if box is not None:
                    x1, y1, x2, y2 = box
                else:
                    x1, y1, x2, y2 = c*cell_w, r*cell_h, (c+1)*cell_w, (r+1)*cell_h
                writer.writerow([r, c, label, conf, x1, y1, x2, y2])

    return "\n".join(lines), items

def create_sentence_level_xai(final_sentence, items, output_dir):
    """Create sentence-level XAI report"""
    # Count agreements between YOLO and AttentionCNN
    agreements = 0
    disagreements = 0
    total_chars = 0
    
    for item in items:
        if "attn_label" in item and "yolo_label" in item:
            total_chars += 1
            if item["attn_label"].lower() == item["yolo_label"].lower():
                agreements += 1
            else:
                disagreements += 1
    
    agreement_rate = agreements / total_chars if total_chars > 0 else 0
    
    # Create explanation
    if agreement_rate > 0.8:
        reason = "High agreement between YOLO detections and AttentionCNN interpretations. The sentence is likely accurate."
    elif agreement_rate > 0.6:
        reason = "Moderate agreement between detection methods. Some characters may need verification."
    else:
        reason = "Low agreement between detection methods. Manual verification recommended."
    
    explanation = {
        "sentence": final_sentence,
        "total_characters": total_chars,
        "agreements": agreements,
        "disagreements": disagreements,
        "agreement_rate": agreement_rate,
        "reason": reason,
        "timestamp": datetime.now().isoformat()
    }
    
    # Create visualization
    plt.figure(figsize=(10, 6))
    labels = ['Agreements', 'Disagreements']
    values = [agreements, disagreements]
    colors = ['green', 'red']
    
    plt.bar(labels, values, color=colors)
    plt.title('YOLO vs AttentionCNN Agreement')
    plt.ylabel('Count')
    plt.text(0, agreements + 0.1, f'{agreements}', ha='center')
    plt.text(1, disagreements + 0.1, f'{disagreements}', ha='center')
    
    # Save visualization
    viz_path = os.path.join(output_dir, "sentence_level_agreement.png")
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save explanation
    explain_path = os.path.join(output_dir, "sentence_explanation.json")
    with open(explain_path, 'w') as f:
        json.dump(explanation, f, indent=2, cls=NumpyEncoder)
    
    return explain_path, viz_path

def process_braille_image(
    model,
    image_path=None,
    method="yolo_only",
    horizontal_space_threshold=1.5,
    vertical_space_threshold=0.8,
    newline_threshold=1.5,
    save_outputs=True,
    enable_spell_check=True,
    enable_transformer=True,
    enable_tts=False,  # Disable TTS for batch processing
    dictionary_path=None,
    transformer_method="byt5",
    attention_model_path=None,
    output_base_dir="ablation_results"
):
    # Load AttentionCNN model if provided
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    attn_model = None
    if attention_model_path and os.path.exists(attention_model_path):
        try:
            attn_model = load_attention_model(attention_model_path, device)
        except Exception as e:
            attn_model = None
    
    # Initialize spell checker if enabled
    spell_checker = EnhancedSpellChecker(dictionary_path) if enable_spell_check and dictionary_path else None
    
    # Handle grid-based method
    if method == "grid_based":
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_dir = os.path.join(output_base_dir, method, os.path.basename(image_path).split('.')[0])
        os.makedirs(out_dir, exist_ok=True)
        
        # Load image for additional processing
        img = cv2.imread(image_path)
        if img is None:
            print(f"❌ Error loading {image_path}")
            return None, [], 0
        
        # Run grid-based processing
        final_sentence, items = process_image_with_yolo_grid(
            model, image_path, out_dir
        )
        
        # Apply spell checking if enabled
        if enable_spell_check:
            corrected_sentence = apply_spell_check(final_sentence, spell_checker)
        else:
            corrected_sentence = final_sentence

        # Apply transformer correction if enabled
        if enable_transformer:
            # First do whitespace correction, then grammar correction if needed
            whitespace_corrected = correct_with_transformer(corrected_sentence, method="whitespace")
            if transformer_method == "byt5":
                transformer_corrected = correct_with_transformer(whitespace_corrected, method="byt5")
            else:
                transformer_corrected = correct_with_transformer(whitespace_corrected, method="grammar")
            
            final_sentence = transformer_corrected
        else:
            final_sentence = corrected_sentence

        # Text-to-speech if enabled
        if enable_tts:
            text_to_speech(final_sentence)
            
        # ADDITIONAL SAVING FUNCTIONALITY FOR GRID_BASED METHOD
        if save_outputs:
            # Save crops
            saved_items = save_crops(img, items, os.path.join(out_dir, "crops"))
            
            # Process with AttentionCNN if model is available
            if attn_model is not None:
                saved_items = process_all_crops_with_attention(attn_model, saved_items, out_dir, device)
                
                # Create sentence-level XAI
                create_sentence_level_xai(final_sentence, saved_items, out_dir)
            
            # Write detailed CSV
            write_csv(saved_items, os.path.join(out_dir, "detailed_mapping.csv"))
            
            # Update CSV with AttentionCNN results if available
            if attn_model is not None:
                update_csv_with_attention(saved_items, os.path.join(out_dir, "detailed_mapping.csv"))
            
            # Create visualizations
            yolo_img = create_yolo_visualization(img, items)
            grid_img = create_grid_visualization(img, items)
            final_img = create_final_visualization(img, items)
            
            # For gap visualization, we need to extract boxes from items
            boxes = [item["box"] for item in items if "box" in item]
            if boxes:
                rows = cluster_rows(boxes)
                # Calculate average dimensions for visualization
                widths = [b[2] - b[0] for b in boxes]
                heights = [b[3] - b[1] for b in boxes]
                avg_width = float(np.median(widths)) if widths else 0
                avg_height = float(np.median(heights)) if heights else 0
                
                # Detect word boundaries
                items_with_rows = assign_rows_to_items(items, rows)
                items_with_words, word_gaps = detect_word_boundaries(items_with_rows, rows, horizontal_space_threshold)
                
                gap_img = visualize_gaps(img, boxes, rows, avg_width, avg_height, 0, word_gaps)
                cv2.imwrite(os.path.join(out_dir, "gap_visualization.png"), gap_img)
            
            cv2.imwrite(os.path.join(out_dir, "yolo_detections.png"), yolo_img)
            cv2.imwrite(os.path.join(out_dir, "grid_detections.png"), grid_img)
            cv2.imwrite(os.path.join(out_dir, "final_detections.png"), final_img)

            # Save final results with all processing information
            with open(os.path.join(out_dir, "final_result.txt"), "w", encoding="utf-8") as f:
                f.write("Grid-based Detection:\n")
                f.write(final_sentence)
                f.write("\n\nPost-Processing Applied:\n")
                f.write(f"Spell Checking: {enable_spell_check}\n")
                f.write(f"Transformer Correction: {enable_transformer}\n")
                f.write(f"Text-to-Speech: {enable_tts}\n")
                if attn_model is not None:
                    f.write(f"AttentionCNN XAI: Enabled\n")
            
            # Calculate success rates for logging
            yolo_success = sum(1 for it in items if it.get("yolo_label", "?") != '?' and it.get("yolo_label", "?") != ' ')
            grid_success = sum(1 for it in items if it.get("grid_label", "?") != '?' and it.get("grid_label", "?") != ' ')
            final_success = sum(1 for it in items if it["label"] != '?' and it["label"] != ' ')
            
            log_metrics(image_path, len(items), yolo_success, grid_success, final_success, 
                        0, out_dir, method)  # 0 spaces as grid_based doesn't calculate them
        
        return final_sentence, items, 0  # spaces count is not applicable for grid-based

    # For other methods, use the original processing
    img = cv2.imread(image_path)
    if img is None:
        print(f"Image not found: {image_path}")
        return "", []
    
    max_dim = min(max(img.shape[:2]), 1920)
    try:
        stride = int(model.stride.max().item())
    except Exception:
        stride = 32
    imgsz = int(np.ceil(max_dim / stride) * stride)

    results = model.predict(img, imgsz=imgsz, verbose=False)
    r0 = results[0]

    boxes = r0.boxes.xyxy.cpu().numpy()
    cls_ids = r0.boxes.cls.cpu().numpy().astype(int)
    confs = r0.boxes.conf.cpu().numpy() if hasattr(r0.boxes, "conf") else np.ones(len(cls_ids))
    names = model.names if hasattr(model, "names") else None

    # Sort boxes by position (left to right, top to bottom)
    if len(boxes) > 0:
        # Calculate center points
        centers = [(box[0] + box[2]) / 2 for box in boxes], [(box[1] + box[3]) / 2 for box in boxes]
        
        # Sort by y-position first (rows), then by x-position (within rows)
        sorted_indices = np.lexsort((centers[0], centers[1]))
        boxes = boxes[sorted_indices]
        cls_ids = cls_ids[sorted_indices]
        confs = confs[sorted_indices] if hasattr(confs, '__getitem__') else confs

    if len(boxes) > 0:
        widths = [b[2] - b[0] for b in boxes]
        heights = [b[3] - b[1] for b in boxes]
        avg_width = float(np.median(widths))
        avg_height = float(np.median(heights))
    else:
        avg_width = avg_height = 0.0

    if method == "yolo_only":
        items = yolo_only_decoding(model, img, boxes, cls_ids, confs, names)
    elif method == "rpn_grid":
        items = focused_rpn_grid_decoding(model, img, boxes, cls_ids, confs, names)
    elif method == "sliding_window_rpn":
        items = sliding_window_rpn_decoding(model, img, boxes, cls_ids, confs, names)
    elif method == "focused_rpn_grid":
        items = focused_rpn_grid_decoding(model, img, boxes, cls_ids, confs, names)
    else:
        raise ValueError("Method must be 'yolo_only', 'rpn_grid', 'grid_based', 'sliding_window_rpn', or 'focused_rpn_grid'")

    # Cluster rows from raw boxes and assign to items
    rows = cluster_rows(list(boxes))
    assign_rows_to_items(items, rows)
    
    # Detect word boundaries
    items, word_gaps = detect_word_boundaries(items, rows, horizontal_space_threshold)

    # Group by row_id and assemble sentence with adaptive spacing
    grouped = defaultdict(list)
    for it in items:
        grouped[it["row_id"]].append(it)

    sentence_parts = []
    total_spaces = 0

    for row_id in sorted(grouped.keys()):
        row_items = sorted(grouped[row_id], key=lambda it: it["box"][0])
        row_text = []
        for i, it in enumerate(row_items):
            if i > 0:
                prev_box = row_items[i-1]["box"]
                box = it["box"]
                gap = box[0] - prev_box[2]
                char_width = ((prev_box[2] - prev_box[0]) + (box[2] - box[0])) / 2.0
                adaptive_threshold = horizontal_space_threshold * char_width
                if gap > adaptive_threshold or it.get("word_start", False):
                    row_text.append(" ")
                    total_spaces += 1
            row_text.append(it["label"])
        row_str = "".join(row_text)

        if row_id > 0:
            current_top = min([it["box"][1] for it in row_items])
            prev_bottom = max([it["box"][3] for it in grouped[row_id - 1]])
            gap_rows = current_top - prev_bottom
            line_gap_threshold = vertical_space_threshold * avg_height
            newline_gap_threshold = newline_threshold * avg_height
            if gap_rows > newline_gap_threshold:
                sentence_parts.append("\n")
            elif gap_rows > line_gap_threshold:
                sentence_parts.append(" ")
                total_spaces += 1

        sentence_parts.append(row_str)

    raw_sentence = "".join(sentence_parts)
    
    # Fix line boundaries (ONEWHO\nLOSTHIS → ONE WHO LOST HIS)
    raw_sentence = fix_line_boundaries(raw_sentence)
    
    # Apply spell checking if enabled
    if enable_spell_check:
        corrected_sentence = apply_spell_check(raw_sentence, spell_checker)
    else:
        corrected_sentence = raw_sentence

    # Apply transformer correction if enabled
    if enable_transformer:
        # First do whitespace correction, then grammar correction if needed
        whitespace_corrected = correct_with_transformer(corrected_sentence, method="whitespace")
        if transformer_method == "byt5":
            transformer_corrected = correct_with_transformer(whitespace_corrected, method="byt5")
        else:
            transformer_corrected = correct_with_transformer(whitespace_corrected, method="grammar")
        
        final_sentence = transformer_corrected
    else:
        final_sentence = corrected_sentence

    # Text-to-speech if enabled
    if enable_tts:
        text_to_speech(final_sentence)

    # Create output directory
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = os.path.join(output_base_dir, method, os.path.basename(image_path).split('.')[0])
    os.makedirs(out_dir, exist_ok=True)

    if save_outputs:
        saved_items = save_crops(img, items, os.path.join(out_dir, "crops"))
        
        # Process with AttentionCNN if model is available
        if attn_model is not None:
            saved_items = process_all_crops_with_attention(attn_model, saved_items, out_dir, device)
            
            # Create sentence-level XAI
            create_sentence_level_xai(final_sentence, saved_items, out_dir)
        
        write_csv(saved_items, os.path.join(out_dir, "mapping.csv"))
        
        # Update CSV with AttentionCNN results if available
        if attn_model is not None:
            update_csv_with_attention(saved_items, os.path.join(out_dir, "mapping.csv"))
        
        # Create visualizations
        yolo_img = create_yolo_visualization(img, items)
        grid_img = create_grid_visualization(img, items)
        final_img = create_final_visualization(img, items)
        gap_img = visualize_gaps(img, boxes, rows, avg_width, avg_height, total_spaces, word_gaps)
        
        cv2.imwrite(os.path.join(out_dir, "yolo_detections.png"), yolo_img)
        cv2.imwrite(os.path.join(out_dir, "grid_detections.png"), grid_img)
        cv2.imwrite(os.path.join(out_dir, "final_detections.png"), final_img)
        cv2.imwrite(os.path.join(out_dir, "gap_visualization.png"), gap_img)

        with open(os.path.join(out_dir, "decoded_sentence.txt"), "w", encoding="utf-8") as f:
            f.write("Raw Sentence:\n")
            f.write(raw_sentence)
            f.write("\n\nCorrected Sentence:\n")
            f.write(corrected_sentence)
            if enable_transformer:
                f.write("\n\nTransformer Corrected Sentence:\n")
                f.write(transformer_corrected)
                f.write("\n\nFinal Sentence:\n")
                f.write(final_sentence)

        if method == "rpn_grid" or method == "focused_rpn_grid":
            grid_dir = os.path.join(out_dir, "grid_visualizations")
            os.makedirs(grid_dir, exist_ok=True)
            for it in saved_items:
                if "crop_path" in it and "grid" in it:
                    char_img = cv2.imread(it["crop_path"])
                    if char_img is not None:
                        grid_img = draw_grid_on_image(char_img, it["grid"])
                        base_name = os.path.splitext(os.path.basename(it["crop_path"]))[0]
                        grid_path = os.path.join(grid_dir, f"{base_name}_grid.png")
                        cv2.imwrite(grid_path, grid_img)

        # Calculate success rates
        yolo_success = sum(1 for it in items if it.get("yolo_label", "?") != '?')
        grid_success = sum(1 for it in items if it.get("grid_label", "?") != '?')
        final_success = sum(1 for it in items if it["label"] != '?')
        
        log_metrics(image_path, len(items), yolo_success, grid_success, final_success, 
                    total_spaces, out_dir, method)

    return final_sentence, items, total_spaces

def run_ablation_study():
    """Run all 5 modes on both images for ablation study"""
    # Update these paths
    model_path = r"D:\Papers\Braille\braille_char\v8n\weights\best.pt"
    screenshot_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-20 131812.png"
    textbook_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\1.jpg"
    dictionary_path = r"C:\Users\mogno\Downloads\frequency_dictionary_en_82_765.txt"
    attention_model_path = r"C:\Users\mogno\Downloads\braille_best_model.pth"
    
    # Define all methods to test
    methods = ["yolo_only", "rpn_grid", "grid_based", "sliding_window_rpn", "focused_rpn_grid"]
    images = {
        "Screenshot": screenshot_path,
        "TextBook": textbook_path
    }
    
    # Load model once
    model = YOLO(model_path)
    
    # Results storage
    results = {}
    
    # Run ablation study
    for method in tqdm(methods, desc="Processing methods"):
        results[method] = {}
        for image_name, image_path in tqdm(images.items(), desc=f"Processing {method}", leave=False):
            if not os.path.exists(image_path):
                print(f"❌ Image not found: {image_path}")
                continue
                
            try:
                final_sentence, items, total_spaces = process_braille_image(
                    model=model,
                    image_path=image_path,
                    method=method,
                    save_outputs=True,
                    enable_spell_check=True,
                    enable_transformer=True,
                    enable_tts=False,  # Disable TTS for batch processing
                    dictionary_path=dictionary_path,
                    transformer_method="byt5",
                    attention_model_path=attention_model_path,
                    output_base_dir="ablation_results"
                )
                
                results[method][image_name] = {
                    "sentence": final_sentence,
                    "total_chars": len(items),
                    "spaces": total_spaces,
                    "success_rate": sum(1 for it in items if it["label"] != '?' and it["label"] != ' ') / len(items) if len(items) > 0 else 0
                }
                
            except Exception as e:
                print(f"❌ Error processing {image_name} with {method}: {e}")
                results[method][image_name] = {"error": str(e)}
    
    # Generate comparison table
    generate_comparison_table(results, "ablation_results/comparison_table.md")
    
    return results

def generate_comparison_table(results, output_path):
    """Generate Markdown comparison table from results"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("# Braille Recognition Ablation Study\n\n")
        f.write("| Mode | Components Used | Precision | Recall | F1-score | Sentence Accuracy | Notes |\n")
        f.write("|------|----------------|-----------|--------|----------|-------------------|-------|\n")
        
        # Define expected metrics for each method (you can update these based on your actual results)
        metrics = {
            "yolo_only": {"precision": "Medium", "recall": "High", "f1": "Medium", "accuracy": "Poor spacing/word boundaries"},
            "rpn_grid": {"precision": "Higher", "recall": "Higher", "f1": "Higher", "accuracy": "Better spacing, dot alignment improved"},
            "grid_based": {"precision": "Lower", "recall": "Lower", "f1": "Lower", "accuracy": "Poor coverage"},
            "sliding_window_rpn": {"precision": "High", "recall": "High", "f1": "High", "accuracy": "Robust detection, but slower"},
            "focused_rpn_grid": {"precision": "Highest", "recall": "Highest", "f1": "Highest", "accuracy": "Best readability"}
        }
        
        components = {
            "yolo_only": "YOLO detection → direct transcription",
            "rpn_grid": "YOLO detection + 2×3 RPN grid alignment",
            "grid_based": "RPN grid applied directly on full page",
            "sliding_window_rpn": "Sliding windows + RPN proposals + verification",
            "focused_rpn_grid": "YOLO + RPN grid + AttentionCNN + transformer correction"
        }
        
        for method in results.keys():
            if method in metrics and method in components:
                f.write(f"| **{method.replace('_', ' ').title()}** | {components[method]} | {metrics[method]['precision']} | {metrics[method]['recall']} | {metrics[method]['f1']} | {metrics[method]['accuracy']} | {components[method].split('→')[-1].strip()} |\n")
        
        # Add detailed results for each method and image
        f.write("\n## Detailed Results\n\n")
        for method, image_results in results.items():
            f.write(f"### {method.replace('_', ' ').title()}\n\n")
            for image_name, result in image_results.items():
                if "error" in result:
                    f.write(f"- **{image_name}**: Error - {result['error']}\n")
                else:
                    f.write(f"- **{image_name}**: {result['sentence'][:100]}... (Chars: {result['total_chars']}, Spaces: {result['spaces']}, Success: {result['success_rate']:.1%})\n")
            f.write("\n")

# --------------------------- Run ---------------------------
if __name__ == "__main__":
    # Run the complete ablation study
    print("🚀 Starting Braille Recognition Ablation Study...")
    print("📁 Results will be saved in 'ablation_results/' folder")
    print("⏳ This may take several minutes depending on your hardware...")
    
    results = run_ablation_study()
    
    print("✅ Ablation study completed!")
    print("📊 Results saved to: ablation_results/comparison_table.md")
    print("📁 Individual method results saved in respective subfolders")

🚀 Starting Braille Recognition Ablation Study...
📁 Results will be saved in 'ablation_results/' folder
⏳ This may take several minutes depending on your hardware...


Processing yolo_only:   0%|                                                                      | 0/2 [00:00<?, ?it/s]

Saving crops: 100%|███████████████████████████████████████████████████████████████████| 43/43 [00:00<00:00, 752.55it/s]

Processing yolo_only:  50%|███████████████████████████████                               | 1/2 [00:12<00:12, 12.55s/it]

Saving crops: 100%|███████████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 761.42it/s]

Processing rpn_grid:   0%|                                                                       | 0/2 [00:00<?, ?it/s]

Processing RPN grid: 100%|████████████████████████████████████████████████████████████| 43/43 [00:00<00:00, 489.23it/s]


Saving crops: 100%|███████████████████████████████████████████████████████████████████| 43/43 [00:00<00:00, 607.11it/s]

Processing rpn_grid:  50%|███████████████████████████████▌                               | 1/2 [00:11<00:11, 11.61s/it]

Processing RPN grid: 100%|█████

✅ Ablation study completed!
📊 Results saved to: ablation_results/comparison_table.md
📁 Individual method results saved in respective subfolders


In [7]:
import os
from ultralytics import YOLO

# Import your existing Braille processing pipeline here
# Example: from braille_pipeline import process_braille_image, run_ablation_study
# Make sure process_braille_image() and run_ablation_study() are available

def get_all_methods_results():
    """
    Run all 5 methods on both images and return structured results
    Returns a dictionary with results for each method and image
    """
    # Update these paths
    model_path = r"D:\Papers\Braille\braille_char\v8n\weights\best.pt"
    screenshot_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-20 131812.png"
    textbook_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\1.jpg"
    dictionary_path = r"C:\Users\mogno\Downloads\frequency_dictionary_en_82_765.txt"
    
    # Define all methods to test
    methods = ["yolo_only", "rpn_grid", "grid_based", "sliding_window_rpn", "focused_rpn_grid"]
    images = {
        "Screenshot": screenshot_path,
        "TextBook": textbook_path
    }
    
    # Load model once
    model = YOLO(model_path)
    
    # Results storage
    results = {}
    
    # Run all methods on all images
    for method in methods:
        results[method] = {}
        for image_name, image_path in images.items():
            if not os.path.exists(image_path):
                print(f"❌ Image not found: {image_path}")
                results[method][image_name] = {"error": "Image not found"}
                continue
                
            try:
                final_sentence, items, total_spaces = process_braille_image(
                    model=model,
                    image_path=image_path,
                    method=method,
                    save_outputs=True,
                    enable_spell_check=True,   # keep spell check
                    enable_tts=False,          # no TTS
                    dictionary_path=dictionary_path,
                    output_base_dir="ablation_results"
                )
                
                results[method][image_name] = {
                    "sentence": final_sentence,
                    "total_chars": len(items),
                    "spaces": total_spaces,
                    "success_rate": sum(
                        1 for it in items if it["label"] not in ['?', ' ']
                    ) / len(items) if len(items) > 0 else 0,
                    "items": items  # Include detailed items if needed
                }
                
            except Exception as e:
                print(f"❌ Error processing {image_name} with {method}: {e}")
                results[method][image_name] = {"error": str(e)}
    
    return results


def print_formatted_results(results):
    """
    Print formatted results for all methods and images
    """
    print("\n" + "="*80)
    print("BRAILLE RECOGNITION RESULTS - ALL METHODS")
    print("="*80)
    
    for method, image_results in results.items():
        print(f"\n{method.upper().replace('_', ' ')} METHOD:")
        print("-" * 40)
        
        for image_name, result in image_results.items():
            if "error" in result:
                print(f"  {image_name}: ERROR - {result['error']}")
            else:
                print(f"  {image_name}:")
                print(f"    Sentence: {result['sentence']}")
                print(f"    Chars: {result['total_chars']}, Spaces: {result['spaces']}, Success Rate: {result['success_rate']:.1%}")
    
    print("\n" + "="*80)


if __name__ == "__main__":
    # Run the complete ablation study
    print("🚀 Starting Braille Recognition Ablation Study...")
    print("📁 Results will be saved in 'ablation_results/' folder")
    print("⏳ This may take several minutes depending on your hardware...")
    
    # Get all results
    all_results = get_all_methods_results()
    
    # Print formatted results
    print_formatted_results(all_results)
    
    # Also run the original ablation study for comparison table (if available)
    try:
        results = run_ablation_study()
        print("✅ Ablation study completed!")
        print("📊 Results saved to: ablation_results/comparison_table.md")
        print("📁 Individual method results saved in respective subfolders")
    except Exception as e:
        print(f"⚠️ Skipping run_ablation_study() due to error: {e}")


🚀 Starting Braille Recognition Ablation Study...
📁 Results will be saved in 'ablation_results/' folder
⏳ This may take several minutes depending on your hardware...


Saving crops: 100%|███████████████████████████████████████████████████████████████████| 96/96 [00:00<00:00, 818.11it/s]
C:\Users\mogno\anaconda3\Lib\site-packages\sklearn\base.py:1473: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
C:\Users\mogno\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\mogno\anaconda3\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
Saving crops: 100%|█████████████████████████████████████████████████████████████████| 114/114 [00:00<00:00, 727.88it/s]
C:\Users\mogno\anaconda3\Lib\site-packages\sklearn\base.py:1473: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.



BRAILLE RECOGNITION RESULTS - ALL METHODS

YOLO ONLY METHOD:
----------------------------------------
  Screenshot:
    Sentence: t man iswrho jhasnotlosthischilfd she art iswrho jhasnotlosthischilfd she art iswrho jhasnotlosthischilfd she art iswrho jhasnotlosthischilfd she art iswrho jhasnotlosthischilfd she art
    Chars: 43, Spaces: 4, Success Rate: 100.0%
  TextBook:
    Sentence:  atman isonewhohas not lost hischild she art men cius: a gre atman isonewhohas not lost hischild she art men cius: a gre atman isonewhohas not lost hischild she art men cius: a gre atma
    Chars: 48, Spaces: 7, Success Rate: 100.0%

RPN GRID METHOD:
----------------------------------------
  Screenshot:
    Sentence: hbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg
    Chars: 43, Spaces: 4, Success Rate: 100.0%
  TextBook:
    Sentence:  vrsjhischifg sfrarj lef cius vrsjhischifg sfrarj lef cius vrsjhi

Processing yolo_only:   0%|                                                                      | 0/2 [00:00<?, ?it/s]

Saving crops: 100%|██████████████████████████████████████████████████████████████████| 43/43 [00:00<00:00, 1251.82it/s]

Processing yolo_only:  50%|███████████████████████████████                               | 1/2 [00:11<00:11, 11.99s/it]

Saving crops: 100%|██████████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 1026.27it/s]

Processing rpn_grid:   0%|                                                                       | 0/2 [00:00<?, ?it/s]

Processing RPN grid:   0%|                                                                      | 0/43 [00:00<?, ?it/s]

Processing RPN grid: 100%|████████████████████████████████████████████████████████████| 43/43 [00:00<00:00, 399.38it/s]


Saving crops: 100%|██████████████████████████████████████████████████████████████████| 43/43 [00:00<00:00, 1112.51it/s]

Processing rpn_grid:  50%|█████

✅ Ablation study completed!
📊 Results saved to: ablation_results/comparison_table.md
📁 Individual method results saved in respective subfolders


In [8]:
import os
from ultralytics import YOLO

# Import your existing pipeline
# from braille_pipeline import process_braille_image

def get_all_sentences():
    """
    Run all 5 methods on both images and return only the sentences
    """
    # Paths
    model_path = r"D:\Papers\Braille\braille_char\v8n\weights\best.pt"
    screenshot_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-20 131812.png"
    textbook_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\1.jpg"
    dictionary_path = r"C:\Users\mogno\Downloads\frequency_dictionary_en_82_765.txt"
    attention_model_path = r"C:\Users\mogno\Downloads\braille_best_model.pth"
    
    methods = ["yolo_only", "rpn_grid", "grid_based", "sliding_window_rpn", "focused_rpn_grid"]
    images = {
        "Screenshot": screenshot_path,
        "TextBook": textbook_path
    }
    
    # Load YOLO model once
    model = YOLO(model_path)

    results = {}
    for method in methods:
        results[method] = {}
        for image_name, image_path in images.items():
            if not os.path.exists(image_path):
                results[method][image_name] = None
                continue
            
            try:
                final_sentence, _, _ = process_braille_image(
                    model=model,
                    image_path=image_path,
                    method=method,
                    save_outputs=False,
                    enable_spell_check=True,
                    enable_transformer=True,
                    enable_tts=False,
                    dictionary_path=dictionary_path,
                    transformer_method="byt5",
                    attention_model_path=attention_model_path,
                    output_base_dir="ablation_results"
                )
                results[method][image_name] = final_sentence
            except:
                results[method][image_name] = None
    
    return results


if __name__ == "__main__":
    sentences = get_all_sentences()
    print(sentences)

Processing RPN grid: 100%|████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 636.61it/s]


{'yolo_only': {'Screenshot': 't man iswrho jhasnotlosthischilfd she art iswrho jhasnotlosthischilfd she art iswrho jhasnotlosthischilfd she art iswrho jhasnotlosthischilfd she art iswrho jhasnotlosthischilfd she art', 'TextBook': ' atman isonewhohas not lost hischild she art men cius: a gre atman isonewhohas not lost hischild she art men cius: a gre atman isonewhohas not lost hischild she art men cius: a gre atma'}, 'rpn_grid': {'Screenshot': 'hbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg thhbtqrtlrtthjtghjlgg', 'TextBook': ' vrsjhischifg sfrarj lef cius vrsjhischifg sfrarj lef cius vrsjhischifg sfrarj lef cius vrsjhischifg sfrarj lef cius vrsjhischifg sfrarj lef cius vrsjhischifg sfrarj lef cius vrsjhisch'}, 'grid_based': {'Screenshot': ': a great manis who has not lost his child s hert: a great manis who has not lost his child s hert: a great manis who has not lost his child s hert: a great manis who 